
# InvoiceIQ – Week 3: Feature Engineering

## Internship Project: Invoice Payment Delay & Cash-Flow Intelligence Platform

### 1. Project Overview

This notebook focuses on Feature Engineering using the cleaned invoice dataset prepared during Week 1 and analyzed during Week 2.

The objective is to transform raw invoice information into meaningful features that can be used by Machine Learning models.

### 2. Project Objectives

The main objectives of this week are:

1. Create time-based features from invoice and due dates.
2. Generate historical customer payment behaviour features.
3. Engineer invoice amount-related features.
4. Encode categorical variables and scale numerical features.
5. Define and validate classification and regression targets.
6. Split the dataset chronologically into training, validation and testing sets.
7. Perform data leakage checks.
8. Prepare a final Machine Learning-ready dataset.

### 3. Prediction Objectives

**Classification:** Predict whether an invoice will be paid late (`paid_late`).

**Regression:** Predict the number of days between the due date and actual payment date (`delay_days`).

### 4. Important Principle

All input features must contain only information available at the defined prediction point.

Future payment information and other post-invoice outcomes must not be used as model inputs.

### 5. Tools and Libraries

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-learn

### 6. Expected Deliverables

- ML-ready dataset
- Feature catalogue
- Chronological data split
- Leakage-check documentation
- README and project explanation


In [1]:
import sys

print(sys.executable)

C:\Users\palar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe


In [1]:
import sys

print(sys.executable)

c:\Users\palar\OneDrive\Desktop\InvoiceIQ\.venv\Scripts\python.exe


In [2]:
import sklearn

print("Scikit-learn version:", sklearn.__version__)

Scikit-learn version: 1.9.1


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

print("All libraries imported successfully!")

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Python executable:", sys.executable)

All libraries imported successfully!
Pandas version: 3.1.0
NumPy version: 2.5.4
Python executable: c:\Users\palar\OneDrive\Desktop\InvoiceIQ\.venv\Scripts\python.exe



## 3. Loading and Inspecting the Cleaned Dataset

In this section, we will locate and load the cleaned invoice dataset prepared during the previous weeks.

The purpose is to ensure that we use the correct processed dataset before beginning feature engineering.

We will inspect:
- Available dataset files
- Dataset dimensions
- Column names
- Data types
- Missing values
- Duplicate records


In [7]:
import os

# Move one level up to the main InvoiceIQ folder
os.chdir("..")

# Display the updated working directory
print("UPDATED PROJECT DIRECTORY:")
print(os.getcwd())

UPDATED PROJECT DIRECTORY:
c:\Users\palar\OneDrive\Desktop\InvoiceIQ


In [8]:
import os

print("AVAILABLE DATA FILES:\n")

for root, dirs, files in os.walk("data"):

    dirs[:] = [
        d for d in dirs
        if d not in [".git", ".venv"]
    ]

    for file in files:
        print(os.path.join(root, file))

AVAILABLE DATA FILES:

data\cleaned\InvoiceIQ_Cleaned_Data.csv
data\raw\test_invoices.csv
data\raw\train_invoices.csv



## 4. Loading and Inspecting the Cleaned Dataset

### 4.1 Dataset Selection

For Week 3, we will use the cleaned invoice dataset generated during Week 1.

**Dataset:** InvoiceIQ_Cleaned_Data.csv

**Location:** data/cleaned/

The original raw training and testing datasets will remain unchanged.

### 4.2 Purpose

Before performing Feature Engineering, we will:

1. Load the cleaned dataset.
2. Identify its dimensions.
3. Inspect column names and data types.
4. Examine sample records.
5. Check missing values.
6. Check duplicate records.
7. Identify columns required for Feature Engineering.


In [9]:

# ==========================================
# LOAD CLEANED INVOICE DATASET
# ==========================================

# Define the dataset path
dataset_path = "data/cleaned/InvoiceIQ_Cleaned_Data.csv"

# Load the cleaned dataset
df = pd.read_csv(dataset_path, low_memory=False)

# Display confirmation
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)


Dataset loaded successfully!
Dataset shape: (2000000, 39)


In [10]:
# Display all column names
print("COLUMN NAMES:")
print(df.columns.tolist())

# Display data types
print("\nDATA TYPES:")
print(df.dtypes)

# Display dataset information
print("\nDATASET INFORMATION:")
df.info()

COLUMN NAMES:
['invoice_id', 'invoice_number', 'invoice_date', 'due_date', 'customer_id', 'customer_name', 'customer_email', 'customer_city', 'customer_state', 'customer_country', 'vendor_id', 'vendor_name', 'product_id', 'product_category', 'product_name', 'quantity', 'unit_price', 'discount_percent', 'tax_percent', 'subtotal', 'discount_amount', 'tax_amount', 'shipping_cost', 'total_amount', 'currency', 'payment_method', 'payment_status', 'payment_date', 'order_channel', 'sales_rep', 'is_overdue', 'payment_date_clean', 'invoice_to_due_days', 'days_to_pay', 'payment_delay_days', 'paid_late_flag', 'is_paid', 'is_open', 'overdue_target']

DATA TYPES:
invoice_id               int64
invoice_number             str
invoice_date               str
due_date                   str
customer_id                str
customer_name              str
customer_email             str
customer_city              str
customer_state             str
customer_country           str
vendor_id                  str
v

In [11]:
# Display first five records
display(df.head())

# Display last five records
display(df.tail())

,invoice_id,invoice_number,invoice_date,due_date,customer_id,customer_name,customer_email,customer_city,customer_state,customer_country,...,sales_rep,is_overdue,payment_date_clean,invoice_to_due_days,days_to_pay,payment_delay_days,paid_late_flag,is_paid,is_open,overdue_target
0,2500001,INV-002500001,2025-12-01,2025-12-16,CUST-033946,James Chen,james.chen33946@example.com,Singapore,Singapore,Singapore,...,Rohan Iyer,0,2025-12-05,15,4.0,-11.0,0.0,1,0,0
1,2500002,INV-002500002,2025-01-04,2025-02-03,CUST-037116,Diya Sharma,diya.sharma37116@example.com,Manchester,England,UK,...,Karen Kumar,0,2025-01-25,30,21.0,-9.0,0.0,1,0,0
2,2500003,INV-002500003,2025-07-07,2025-08-06,CUST-027731,Reyansh Nair,reyansh.nair27731@example.com,London,England,UK,...,Richard Miller,0,2025-07-21,30,14.0,-16.0,0.0,1,0,0
3,2500004,INV-002500004,2025-02-17,2025-03-19,CUST-047646,Arjun Singh,arjun.singh47646@example.com,San Francisco,California,USA,...,Krishna Sharma,0,2025-03-11,30,22.0,-8.0,0.0,1,0,0
4,2500005,INV-002500005,2024-05-09,2024-05-24,CUST-015519,Thomas Wang,thomas.wang15519@example.com,Seattle,Washington,USA,...,Rohan Nair,1,NaN,15,NaN,NaN,NaN,0,1,1


,invoice_id,invoice_number,invoice_date,due_date,customer_id,customer_name,customer_email,customer_city,customer_state,customer_country,...,sales_rep,is_overdue,payment_date_clean,invoice_to_due_days,days_to_pay,payment_delay_days,paid_late_flag,is_paid,is_open,overdue_target
1999995,4499996,INV-004499996,2024-09-14,2024-11-13,CUST-012695,Susan Liu,susan.liu12695@example.com,Houston,Texas,USA,...,William Rodriguez,0,2024-10-15,60,31.0,-29.0,0.0,1,0,0
1999996,4499997,INV-004499997,2025-07-17,2025-09-15,CUST-054903,Linda Reddy,linda.reddy54903@example.com,Tokyo,Tokyo,Japan,...,Susan Iyer,0,2025-08-23,60,37.0,-23.0,0.0,1,0,0
1999997,4499998,INV-004499998,2025-02-06,2025-04-07,CUST-024764,Vivaan Chen,vivaan.chen24764@example.com,Bengaluru,Karnataka,India,...,Elizabeth Brown,0,2025-03-27,60,49.0,-11.0,0.0,1,0,0
1999998,4499999,INV-004499999,2024-11-28,2024-12-13,CUST-011635,Ananya Watanabe,ananya.watanabe11635@example.com,Boston,Massachusetts,USA,...,Vihaan Verma,0,2024-12-12,15,14.0,-1.0,0.0,1,0,0
1999999,4500000,INV-004500000,2025-10-25,2025-12-24,CUST-041037,Kim Garcia,kim.garcia41037@example.com,Ahmedabad,Gujarat,India,...,Hiro Zhang,0,NaN,60,NaN,NaN,NaN,0,1,0


In [12]:
# Count completely duplicated rows
duplicate_count = df.duplicated().sum()

print("Total duplicate records:", duplicate_count)

Total duplicate records: 0


In [13]:

# ==========================================
# MISSING VALUE ANALYSIS
# ==========================================

missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().sum() / len(df)
    ) * 100
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    by="Missing_Count",
    ascending=False
)

print("COLUMNS WITH MISSING VALUES:")
display(missing_summary)

print("TOTAL MISSING VALUES:", df.isnull().sum().sum())


COLUMNS WITH MISSING VALUES:


,Missing_Count,Missing_Percentage
payment_date_clean,641181,32.05905
days_to_pay,641181,32.05905
payment_delay_days,641181,32.05905
paid_late_flag,641181,32.05905


TOTAL MISSING VALUES: 2564724


In [14]:

# ==========================================
# PAYMENT STATUS ANALYSIS
# ==========================================

print("PAYMENT STATUS DISTRIBUTION:")
display(df["payment_status"].value_counts(dropna=False))

print("\nPAID FLAG DISTRIBUTION:")
display(df["is_paid"].value_counts(dropna=False))

print("\nOPEN FLAG DISTRIBUTION:")
display(df["is_open"].value_counts(dropna=False))

print("\nPAID LATE FLAG DISTRIBUTION:")
display(df["paid_late_flag"].value_counts(dropna=False))

print("\nOVERDUE TARGET DISTRIBUTION:")
display(df["overdue_target"].value_counts(dropna=False))


PAYMENT STATUS DISTRIBUTION:


payment_status
Paid         1358819
Pending       341015
Overdue       199777
Cancelled     100389
Name: count, dtype: int64


PAID FLAG DISTRIBUTION:


is_paid
1    1358819
0     641181
Name: count, dtype: int64


OPEN FLAG DISTRIBUTION:


is_open
0    1459208
1     540792
Name: count, dtype: int64


PAID LATE FLAG DISTRIBUTION:


paid_late_flag
0.0    1358819
NaN     641181
Name: count, dtype: int64


OVERDUE TARGET DISTRIBUTION:


overdue_target
0    1800223
1     199777
Name: count, dtype: int64

In [15]:
# ==========================================
# DATE RANGE ANALYSIS
# ==========================================

date_columns = [
    "invoice_date",
    "due_date",
    "payment_date"
]

for column in date_columns:

    converted_dates = pd.to_datetime(
        df[column],
        errors="coerce"
    )

    print(f"\n{column.upper()}")
    print("Earliest:", converted_dates.min())
    print("Latest:", converted_dates.max())
    print("Invalid or missing:", converted_dates.isna().sum())


INVOICE_DATE
Earliest: 2024-01-01 00:00:00
Latest: 2025-12-31 00:00:00
Invalid or missing: 0

DUE_DATE
Earliest: 2024-01-16 00:00:00
Latest: 2026-03-01 00:00:00
Invalid or missing: 0

PAYMENT_DATE
Earliest: 2024-01-01 00:00:00
Latest: 2026-03-01 00:00:00
Invalid or missing: 0


In [16]:
# ==========================================
# VERIFY PAYMENT DATA AND TARGET COLUMNS
# ==========================================

print("1. PAYMENT STATUS DISTRIBUTION")
display(df["payment_status"].value_counts(dropna=False))

print("\n2. IS_PAID DISTRIBUTION")
display(df["is_paid"].value_counts(dropna=False))

print("\n3. IS_OPEN DISTRIBUTION")
display(df["is_open"].value_counts(dropna=False))

print("\n4. PAID_LATE_FLAG DISTRIBUTION")
display(df["paid_late_flag"].value_counts(dropna=False))

print("\n5. PAYMENT DELAY DISTRIBUTION")
display(df["payment_delay_days"].describe())

print("\n6. PAYMENT DELAY VALUE COUNTS")
display(df["payment_delay_days"].value_counts(dropna=False).head(15))

print("\n7. IS_PAID VS PAYMENT_DATE_CLEAN")
display(pd.crosstab(
    df["is_paid"],
    df["payment_date_clean"].isna(),
    margins=True
))

print("\n8. IS_OPEN VS PAYMENT_DATE_CLEAN")
display(pd.crosstab(
    df["is_open"],
    df["payment_date_clean"].isna(),
    margins=True
))

print("\n9. ORIGINAL VS CLEANED PAYMENT DATE")
display(
    df.loc[
        df["payment_date_clean"].isna(),
        ["payment_date", "payment_date_clean", "payment_status", "is_paid", "is_open"]
    ].head(10)
)

1. PAYMENT STATUS DISTRIBUTION


payment_status
Paid         1358819
Pending       341015
Overdue       199777
Cancelled     100389
Name: count, dtype: int64


2. IS_PAID DISTRIBUTION


is_paid
1    1358819
0     641181
Name: count, dtype: int64


3. IS_OPEN DISTRIBUTION


is_open
0    1459208
1     540792
Name: count, dtype: int64

... (8 outputs left out)


8. IS_OPEN VS PAYMENT_DATE_CLEAN


payment_date_clean,False,True,All
is_open,,,
0,1358819,100389,1459208
1,0,540792,540792
All,1358819,641181,2000000



9. ORIGINAL VS CLEANED PAYMENT DATE


,payment_date,payment_date_clean,payment_status,is_paid,is_open
4,2024-05-24,NaN,Overdue,0,1
5,2025-02-08,NaN,Overdue,0,1
8,2026-01-05,NaN,Overdue,0,1
10,2024-10-01,NaN,Pending,0,1
12,2025-05-20,NaN,Pending,0,1
13,2024-08-27,NaN,Cancelled,0,0
15,2025-03-26,NaN,Cancelled,0,0
22,2024-09-06,NaN,Pending,0,1
27,2024-11-26,NaN,Overdue,0,1
29,2024-08-15,NaN,Cancelled,0,0


Title: Dataset Target Validation and Data Quality Investigation

During initial dataset validation, the payment delay distribution showed no positive values, indicating that none of the invoices with valid cleaned payment dates were recorded as paid after their due dates. The paid_late_flag column also contains only zero values and missing values, with no positive class examples. Additionally, the raw payment_date column contains dates for some unpaid and cancelled invoices, whereas payment_date_clean is missing for these records. Further validation is required before defining the final supervised learning targets.

In [17]:

# ==========================================
# WEEK 3 - TARGET VALIDATION
# ==========================================

# 1. Convert relevant columns into datetime
df["invoice_date"] = pd.to_datetime(
    df["invoice_date"], errors="coerce"
)

df["due_date"] = pd.to_datetime(
    df["due_date"], errors="coerce"
)

df["payment_date"] = pd.to_datetime(
    df["payment_date"], errors="coerce"
)

df["payment_date_clean"] = pd.to_datetime(
    df["payment_date_clean"], errors="coerce"
)

# 2. Recalculate payment delay independently
recalculated_delay = (
    df["payment_date_clean"] - df["due_date"]
).dt.days

# 3. Display recalculated delay distribution
print("RECALCULATED PAYMENT DELAY:")
display(recalculated_delay.describe())

print("\nPOSITIVE PAYMENT DELAYS:")
print((recalculated_delay > 0).sum())

print("\nZERO PAYMENT DELAYS:")
print((recalculated_delay == 0).sum())

print("\nNEGATIVE PAYMENT DELAYS:")
print((recalculated_delay < 0).sum())

# 4. Compare existing and recalculated delays
comparison = pd.DataFrame({
    "Existing_Delay": df["payment_delay_days"],
    "Recalculated_Delay": recalculated_delay
})

print("\nDELAY CALCULATION MISMATCHES:")
print(
    (
        comparison["Existing_Delay"]
        != comparison["Recalculated_Delay"]
    ).sum()
)

# 5. Check payment status against paid flag
print("\nPAYMENT STATUS VS IS_PAID:")
display(pd.crosstab(
    df["payment_status"],
    df["is_paid"],
    margins=True
))

# 6. Check raw payment dates against cleaned dates
print("\nRAW PAYMENT DATE AVAILABILITY:")
display(pd.crosstab(
    df["payment_status"],
    df["payment_date"].notna(),
    margins=True
))

print("\nCLEANED PAYMENT DATE AVAILABILITY:")
display(pd.crosstab(
    df["payment_status"],
    df["payment_date_clean"].notna(),
    margins=True
))

# 7. Check whether any paid invoice has a positive delay
print("\nPAID INVOICES WITH POSITIVE DELAY:")
print(
    (
        (df["is_paid"] == 1)
        & (recalculated_delay > 0)
    ).sum()
)



RECALCULATED PAYMENT DELAY:


count    1.358819e+06
mean    -1.501869e+01
std      1.216216e+01
min     -6.000000e+01
25%     -2.200000e+01
50%     -1.200000e+01
75%     -6.000000e+00
max      0.000000e+00
dtype: float64


POSITIVE PAYMENT DELAYS:
0

ZERO PAYMENT DELAYS:
53752

NEGATIVE PAYMENT DELAYS:
1305067

DELAY CALCULATION MISMATCHES:
641181

PAYMENT STATUS VS IS_PAID:


is_paid,0,1,All
payment_status,,,
Cancelled,100389,0,100389
Overdue,199777,0,199777
Paid,0,1358819,1358819
Pending,341015,0,341015
All,641181,1358819,2000000



RAW PAYMENT DATE AVAILABILITY:


payment_date,True,All
payment_status,,
Cancelled,100389,100389
Overdue,199777,199777
Paid,1358819,1358819
Pending,341015,341015
All,2000000,2000000



CLEANED PAYMENT DATE AVAILABILITY:


payment_date_clean,False,True,All
payment_status,,,
Cancelled,100389,0,100389
Overdue,199777,0,199777
Paid,0,1358819,1358819
Pending,341015,0,341015
All,641181,1358819,2000000



PAID INVOICES WITH POSITIVE DELAY:
0


In [18]:

# ==========================================
# VERIFY PAYMENT DELAY CALCULATION
# ==========================================

valid_rows = (
    df["payment_delay_days"].notna()
    & recalculated_delay.notna()
)

actual_mismatches = (
    df.loc[valid_rows, "payment_delay_days"]
    != recalculated_delay.loc[valid_rows]
)

print("Rows compared:", valid_rows.sum())
print("Actual mismatches:", actual_mismatches.sum())

# Check whether raw payment dates match cleaned dates
raw_vs_clean = (
    df["payment_date"] == df["payment_date_clean"]
)

print("\nRAW VS CLEANED PAYMENT DATE:")
display(pd.crosstab(
    df["payment_status"],
    raw_vs_clean,
    margins=True
))

# Inspect examples of unpaid invoices with raw dates
print("\nUNPAID INVOICE DATE EXAMPLES:")
display(
    df.loc[
        df["is_paid"] == 0,
        [
            "invoice_id",
            "invoice_date",
            "due_date",
            "payment_date",
            "payment_date_clean",
            "payment_status",
            "is_paid"
        ]
    ].head(10)
)


Rows compared: 1358819
Actual mismatches: 0

RAW VS CLEANED PAYMENT DATE:


col_0,False,True,All
payment_status,,,
Cancelled,100389,0,100389
Overdue,199777,0,199777
Paid,0,1358819,1358819
Pending,341015,0,341015
All,641181,1358819,2000000



UNPAID INVOICE DATE EXAMPLES:


,invoice_id,invoice_date,due_date,payment_date,payment_date_clean,payment_status,is_paid
4,2500005,2024-05-09,2024-05-24,2024-05-24,NaT,Overdue,0
5,2500006,2025-01-09,2025-02-08,2025-02-08,NaT,Overdue,0
8,2500009,2025-12-06,2026-01-05,2026-01-05,NaT,Overdue,0
10,2500011,2024-08-17,2024-10-01,2024-10-01,NaT,Pending,0
12,2500013,2025-04-20,2025-05-20,2025-05-20,NaT,Pending,0
13,2500014,2024-07-28,2024-08-27,2024-08-27,NaT,Cancelled,0
15,2500016,2025-02-09,2025-03-26,2025-03-26,NaT,Cancelled,0
22,2500023,2024-08-07,2024-09-06,2024-09-06,NaT,Pending,0
27,2500028,2024-10-12,2024-11-26,2024-11-26,NaT,Overdue,0
29,2500030,2024-07-16,2024-08-15,2024-08-15,NaT,Cancelled,0


Final Data Quality Findings and Target Definition

The cleaned dataset contains 2,000,000 invoice records, of which 1,358,819 are marked as Paid and 641,181 are unpaid or cancelled.

The payment_date_clean column contains valid payment dates only for paid invoices. The raw payment_date column contains dates even for unpaid and cancelled invoices; sample records show these dates matching the due date. Therefore, raw payment dates are not considered reliable evidence of actual payment.

The existing payment_delay_days calculation was independently verified for all 1,358,819 paid invoices, with zero actual mismatches.

The payment delay distribution contains 1,305,067 early payments, 53,752 payments made exactly on the due date, and zero late payments.

Consequently, the current dataset cannot support meaningful positive-class training for the paid_late classification objective. The delay_days regression target can describe early and on-time payments but has no examples of late payments.

Unpaid and cancelled invoices will not be assigned invented payment outcomes. Their payment targets will remain missing until actual payment outcomes are available.

Task A – Time-Based Feature Engineering

Time-based features are created using invoice_date and due_date. The prediction point is defined as the invoice issuance date. Only information available at that point is used. The month-end flag is set to 1 when the invoice date is the final calendar day of its month.

In [21]:

# ==========================================
# TASK A - TIME-BASED FEATURE ENGINEERING
# ==========================================

# 1. Extract invoice month
df["invoice_month"] = df["invoice_date"].dt.month

# 2. Extract invoice quarter
df["invoice_quarter"] = df["invoice_date"].dt.quarter

# 3. Extract weekday
# Monday = 0, Sunday = 6
df["invoice_weekday"] = df["invoice_date"].dt.day_of_week

# 4. Identify month-end invoices
df["is_month_end"] = (
    df["invoice_date"].dt.is_month_end
).astype("int8")

# 5. Calculate payment term length
df["payment_term_days"] = (
    df["due_date"] - df["invoice_date"]
).dt.days

# 6. Define prediction date as invoice date
prediction_date = df["invoice_date"]

# Calculate days until due
df["days_until_due"] = (
    df["due_date"] - prediction_date
).dt.days

# 7. Display newly created features
time_features = [
    "invoice_date",
    "due_date",
    "invoice_month",
    "invoice_quarter",
    "invoice_weekday",
    "is_month_end",
    "payment_term_days",
    "days_until_due"
]

display(df[time_features].head(10))

print("\nTIME FEATURE CREATION COMPLETED!")


,invoice_date,due_date,invoice_month,invoice_quarter,invoice_weekday,is_month_end,payment_term_days,days_until_due
0,2025-12-01,2025-12-16,12,4,0,0,15,15
1,2025-01-04,2025-02-03,1,1,5,0,30,30
2,2025-07-07,2025-08-06,7,3,0,0,30,30
3,2025-02-17,2025-03-19,2,1,0,0,30,30
4,2024-05-09,2024-05-24,5,2,3,0,15,15
5,2025-01-09,2025-02-08,1,1,3,0,30,30
6,2025-11-19,2025-12-04,11,4,2,0,15,15
7,2025-12-11,2025-12-26,12,4,3,0,15,15
8,2025-12-06,2026-01-05,12,4,5,0,30,30
9,2025-03-25,2025-04-09,3,1,1,0,15,15



TIME FEATURE CREATION COMPLETED!


In [22]:

# ==========================================
# TIME FEATURE VALIDATION
# ==========================================

print("MISSING VALUES IN TIME FEATURES:")

display(df[
    [
        "invoice_month",
        "invoice_quarter",
        "invoice_weekday",
        "is_month_end",
        "payment_term_days",
        "days_until_due"
    ]
].isnull().sum())

print("\nINVOICE MONTH DISTRIBUTION:")
display(df["invoice_month"].value_counts().sort_index())

print("\nINVOICE QUARTER DISTRIBUTION:")
display(df["invoice_quarter"].value_counts().sort_index())

print("\nWEEKDAY DISTRIBUTION:")
display(df["invoice_weekday"].value_counts().sort_index())

print("\nMONTH-END DISTRIBUTION:")
display(df["is_month_end"].value_counts())

print("\nPAYMENT TERM SUMMARY:")
display(df["payment_term_days"].describe())

print("\nCHECKING REDUNDANT FEATURES:")
print(
    "Identical values:",
    (
        df["payment_term_days"]
        == df["days_until_due"]
    ).all()
)


MISSING VALUES IN TIME FEATURES:


invoice_month        0
invoice_quarter      0
invoice_weekday      0
is_month_end         0
payment_term_days    0
days_until_due       0
dtype: int64


INVOICE MONTH DISTRIBUTION:


invoice_month
1     169633
2     155784
3     169800
4     163522
5     169977
6     164369
7     169803
8     169730
9     163970
10    169826
11    163760
12    169826
Name: count, dtype: int64


INVOICE QUARTER DISTRIBUTION:


invoice_quarter
1    495217
2    497868
3    503503
4    503412
Name: count, dtype: int64

... (3 outputs left out)

is_month_end
0    1934675
1      65325
Name: count, dtype: int64


PAYMENT TERM SUMMARY:


count    2.000000e+06
mean     3.001804e+01
std      1.423578e+01
min      1.500000e+01
25%      1.500000e+01
50%      3.000000e+01
75%      4.500000e+01
max      6.000000e+01
Name: payment_term_days, dtype: float64


CHECKING REDUNDANT FEATURES:
Identical values: True


Task B – Customer Historical Behaviour Features

Customer historical behaviour features capture the payment patterns and transaction history of customers. These features are calculated chronologically using only information available before the current invoice's prediction date.

Previous payment delay and late-payment rate use only earlier invoices that had already been paid before the current invoice was issued. Previous invoice count and average invoice amount use earlier invoices issued before the current invoice.

Customers without eligible historical records will receive zero for the previous invoice count and missing values for historical averages and rates. These missing values indicate the absence of historical information, not an average payment performance of zero.

In [23]:

# ==========================================
# TASK B - CUSTOMER HISTORY PRE-CHECK
# ==========================================

print("TOTAL UNIQUE CUSTOMERS:")
print(df["customer_id"].nunique())

print("\nMISSING CUSTOMER IDs:")
print(df["customer_id"].isna().sum())

print("\nINVOICE DATE RANGE:")
print("Earliest:", df["invoice_date"].min())
print("Latest:", df["invoice_date"].max())

print("\nINVOICES PER CUSTOMER:")
customer_invoice_counts = df.groupby(
    "customer_id"
).size()

display(customer_invoice_counts.describe())

print("\nCUSTOMERS WITH REPEATED INVOICES:")
print((customer_invoice_counts > 1).sum())

print("\nSAMPLE CUSTOMER HISTORY:")
sample_customer = df["customer_id"].value_counts().index[0]

display(
    df.loc[
        df["customer_id"] == sample_customer,
        [
            "customer_id",
            "invoice_id",
            "invoice_date",
            "due_date",
            "total_amount",
            "payment_date_clean",
            "payment_delay_days",
            "paid_late_flag"
        ]
    ].sort_values("invoice_date").head(15)
)


TOTAL UNIQUE CUSTOMERS:
60000

MISSING CUSTOMER IDs:
0

INVOICE DATE RANGE:
Earliest: 2024-01-01 00:00:00
Latest: 2025-12-31 00:00:00

INVOICES PER CUSTOMER:


count    60000.000000
mean        33.333333
std          5.778989
min          8.000000
25%         29.000000
50%         33.000000
75%         37.000000
max         59.000000
dtype: float64


CUSTOMERS WITH REPEATED INVOICES:
60000

SAMPLE CUSTOMER HISTORY:


,customer_id,invoice_id,invoice_date,due_date,total_amount,payment_date_clean,payment_delay_days,paid_late_flag
915937,CUST-015153,3415938,2024-01-05,2024-02-04,79099.24,2024-01-29,-6.0,0.0
1190331,CUST-015153,3690332,2024-01-07,2024-02-21,17298.25,2024-02-01,-20.0,0.0
1948732,CUST-015153,4448733,2024-01-28,2024-02-27,749.90,2024-02-22,-5.0,0.0
1439858,CUST-015153,3939859,2024-02-09,2024-02-24,52918.80,2024-02-13,-11.0,0.0
1255882,CUST-015153,3755883,2024-02-12,2024-03-13,40711.89,NaT,NaN,NaN
1242990,CUST-015153,3742991,2024-02-20,2024-03-21,38767.15,NaT,NaN,NaN
850681,CUST-015153,3350682,2024-02-24,2024-04-09,23971.69,NaT,NaN,NaN
770928,CUST-015153,3270929,2024-03-16,2024-04-15,3532.85,NaT,NaN,NaN
1389009,CUST-015153,3889010,2024-03-28,2024-05-12,62697.82,2024-04-14,-28.0,0.0
288044,CUST-015153,2788045,2024-03-28,2024-04-27,2679.11,NaT,NaN,NaN


In [24]:

# ==========================================
# CHECK HISTORICAL DATE VALIDITY
# ==========================================

paid_mask = df["payment_date_clean"].notna()

payment_before_invoice = (
    paid_mask
    & (df["payment_date_clean"] < df["invoice_date"])
)

print(
    "Paid invoices with payment date before invoice date:",
    payment_before_invoice.sum()
)

print(
    "Paid invoices with payment date equal to invoice date:",
    (
        paid_mask
        & (df["payment_date_clean"] == df["invoice_date"])
    ).sum()
)

print(
    "Invoices with missing total amount:",
    df["total_amount"].isna().sum()
)

print(
    "Invoices with zero or negative total amount:",
    (df["total_amount"] <= 0).sum()
)



Paid invoices with payment date before invoice date: 0
Paid invoices with payment date equal to invoice date: 54174
Invoices with missing total amount: 0
Invoices with zero or negative total amount: 0


In [25]:

# ==========================================
# TASK B - PART 1
# PREVIOUS INVOICE COUNT AND AVERAGE AMOUNT
# ==========================================

# Select only the columns required for this calculation
invoice_history = df[
    [
        "customer_id",
        "invoice_date",
        "total_amount"
    ]
].copy()

# Sort invoices customer-wise and chronologically
invoice_history = invoice_history.sort_values(
    ["customer_id", "invoice_date"],
    kind="mergesort"
)

# Calculate the number of earlier invoices
customer_cumulative_count = (
    invoice_history.groupby(
        "customer_id",
        sort=False
    ).cumcount()
)

# Calculate the position within the same customer's same-day invoices
same_day_count = (
    invoice_history.groupby(
        ["customer_id", "invoice_date"],
        sort=False
    ).cumcount()
)

# Exclude invoices issued on the current date
previous_invoice_count = (
    customer_cumulative_count - same_day_count
)

# Calculate cumulative invoice amounts customer-wise
customer_cumulative_amount = (
    invoice_history.groupby(
        "customer_id",
        sort=False
    )["total_amount"].cumsum()
)

# Calculate cumulative amounts within the same date
same_day_cumulative_amount = (
    invoice_history.groupby(
        ["customer_id", "invoice_date"],
        sort=False
    )["total_amount"].cumsum()
)

# Remove current-day invoice amounts
previous_invoice_amount_sum = (
    customer_cumulative_amount
    - same_day_cumulative_amount
)

# Calculate historical average amount
previous_average_amount = np.divide(
    previous_invoice_amount_sum.to_numpy(),
    previous_invoice_count.to_numpy(),
    out=np.full(len(invoice_history), np.nan),
    where=previous_invoice_count.to_numpy() > 0
)

# Assign results back to the original dataframe
# Original row indexes are preserved
df.loc[
    invoice_history.index,
    "customer_previous_invoice_count"
] = previous_invoice_count.to_numpy()

df.loc[
    invoice_history.index,
    "customer_previous_avg_amount"
] = previous_average_amount

print("PREVIOUS INVOICE FEATURES CREATED SUCCESSFULLY!")


PREVIOUS INVOICE FEATURES CREATED SUCCESSFULLY!


In [26]:

# ==========================================
# VALIDATE PREVIOUS INVOICE FEATURES
# ==========================================

print("MISSING VALUES:")
display(
    df[
        [
            "customer_previous_invoice_count",
            "customer_previous_avg_amount"
        ]
    ].isnull().sum()
)

print("\nPREVIOUS INVOICE COUNT SUMMARY:")
display(
    df["customer_previous_invoice_count"].describe()
)

print("\nPREVIOUS AVERAGE AMOUNT SUMMARY:")
display(
    df["customer_previous_avg_amount"].describe()
)

print("\nINVOICES WITHOUT PREVIOUS CUSTOMER HISTORY:")
print(
    (df["customer_previous_invoice_count"] == 0).sum()
)

print("\nSAMPLE CUSTOMER HISTORICAL FEATURES:")

display(
    df.loc[
        df["customer_id"] == sample_customer,
        [
            "customer_id",
            "invoice_date",
            "total_amount",
            "customer_previous_invoice_count",
            "customer_previous_avg_amount"
        ]
    ].sort_values("invoice_date").head(15)
)


MISSING VALUES:


customer_previous_invoice_count        0
customer_previous_avg_amount       61379
dtype: int64


PREVIOUS INVOICE COUNT SUMMARY:


count    2.000000e+06
mean     1.664481e+01
std      1.045248e+01
min      0.000000e+00
25%      8.000000e+00
50%      1.600000e+01
75%      2.500000e+01
max      5.800000e+01
Name: customer_previous_invoice_count, dtype: float64


PREVIOUS AVERAGE AMOUNT SUMMARY:


count    1.938621e+06
mean     3.396527e+04
std      1.055444e+04
min      1.040000e+01
25%      2.829852e+04
50%      3.354094e+04
75%      3.897090e+04
max      1.577702e+05
Name: customer_previous_avg_amount, dtype: float64


INVOICES WITHOUT PREVIOUS CUSTOMER HISTORY:
61379

SAMPLE CUSTOMER HISTORICAL FEATURES:


,customer_id,invoice_date,total_amount,customer_previous_invoice_count,customer_previous_avg_amount
915937,CUST-015153,2024-01-05,79099.24,0.0,NaN
1190331,CUST-015153,2024-01-07,17298.25,1.0,79099.240000
1948732,CUST-015153,2024-01-28,749.90,2.0,48198.745000
1439858,CUST-015153,2024-02-09,52918.80,3.0,32382.463333
1255882,CUST-015153,2024-02-12,40711.89,4.0,37516.547500
1242990,CUST-015153,2024-02-20,38767.15,5.0,38155.616000
850681,CUST-015153,2024-02-24,23971.69,6.0,38257.538333
770928,CUST-015153,2024-03-16,3532.85,7.0,36216.702857
1389009,CUST-015153,2024-03-28,62697.82,8.0,32131.221250
288044,CUST-015153,2024-03-28,2679.11,8.0,32131.221250


In [27]:

# ==========================================
# TASK B - PART 2
# PREVIOUS PAYMENT BEHAVIOUR
# ==========================================

# Convert required columns into NumPy arrays
invoice_dates = df["invoice_date"].to_numpy(
    dtype="datetime64[ns]"
)

payment_dates = df["payment_date_clean"].to_numpy(
    dtype="datetime64[ns]"
)

payment_delays = pd.to_numeric(
    df["payment_delay_days"],
    errors="coerce"
).to_numpy(dtype="float64")


# Create empty arrays for the new features
number_of_rows = len(df)

previous_avg_delay = np.full(
    number_of_rows,
    np.nan,
    dtype="float64"
)

previous_late_rate = np.full(
    number_of_rows,
    np.nan,
    dtype="float64"
)


# Group invoice row positions by customer
customer_groups = df.groupby(
    "customer_id",
    sort=False
).indices


# Process each customer separately
for positions in customer_groups.values():

    # Get this customer's payment information
    group_payment_dates = payment_dates[positions]
    group_delays = payment_delays[positions]

    # Keep only valid completed payments
    valid_paid = (
        ~np.isnat(group_payment_dates)
        & np.isfinite(group_delays)
    )

    # Skip customers without any completed payments
    if not valid_paid.any():
        continue

    # Extract valid payment dates and delays
    event_dates = group_payment_dates[valid_paid]
    event_delays = group_delays[valid_paid]

    # Sort payment events chronologically
    order = np.argsort(event_dates)

    event_dates = event_dates[order]
    event_delays = event_delays[order]

    # Combine payments completed on the same date
    unique_dates, inverse = np.unique(
        event_dates,
        return_inverse=True
    )

    # Count completed payments on each date
    daily_paid_count = np.bincount(
        inverse
    )

    # Calculate total payment delay on each date
    daily_delay_sum = np.bincount(
        inverse,
        weights=event_delays
    )

    # Count late payments on each date
    daily_late_count = np.bincount(
        inverse,
        weights=(event_delays > 0).astype(float)
    )

    # Calculate cumulative historical totals
    cumulative_paid_count = np.cumsum(
        daily_paid_count
    )

    cumulative_delay_sum = np.cumsum(
        daily_delay_sum
    )

    cumulative_late_count = np.cumsum(
        daily_late_count
    )

    # Find payment events strictly before each invoice date
    current_invoice_dates = invoice_dates[positions]

    previous_payment_index = (
        np.searchsorted(
            unique_dates,
            current_invoice_dates,
            side="left"
        ) - 1
    )

    # Identify invoices with at least one previous payment
    has_payment_history = previous_payment_index >= 0

    # Calculate historical averages and late-payment rates
    valid_positions = positions[has_payment_history]

    valid_history_index = previous_payment_index[
        has_payment_history
    ]

    previous_avg_delay[valid_positions] = (
        cumulative_delay_sum[valid_history_index]
        / cumulative_paid_count[valid_history_index]
    )

    previous_late_rate[valid_positions] = (
        cumulative_late_count[valid_history_index]
        / cumulative_paid_count[valid_history_index]
    )


# Assign calculated features to the original dataframe
df["customer_previous_avg_delay"] = previous_avg_delay

df["customer_previous_late_rate"] = previous_late_rate


print("CUSTOMER PAYMENT HISTORY FEATURES CREATED SUCCESSFULLY!")


CUSTOMER PAYMENT HISTORY FEATURES CREATED SUCCESSFULLY!


In [28]:

# ==========================================
# VALIDATE CUSTOMER PAYMENT HISTORY
# ==========================================

history_columns = [
    "customer_previous_avg_delay",
    "customer_previous_late_rate"
]

print("MISSING VALUES:")
display(df[history_columns].isnull().sum())


print("\nPREVIOUS AVERAGE PAYMENT DELAY:")
display(
    df["customer_previous_avg_delay"].describe()
)


print("\nPREVIOUS LATE PAYMENT RATE:")
display(
    df["customer_previous_late_rate"].describe()
)


print("\nINVOICES WITH NO PREVIOUS PAYMENT HISTORY:")

print(
    df["customer_previous_avg_delay"].isna().sum()
)


print("\nDISTRIBUTION OF PREVIOUS LATE PAYMENT RATE:")

display(
    df["customer_previous_late_rate"].value_counts(
        dropna=False
    ).head(10)
)


MISSING VALUES:


customer_previous_avg_delay    126024
customer_previous_late_rate    126024
dtype: int64


PREVIOUS AVERAGE PAYMENT DELAY:


count    1.873976e+06
mean    -1.525847e+01
std      5.159280e+00
min     -6.000000e+01
25%     -1.761538e+01
50%     -1.487500e+01
75%     -1.233333e+01
max      0.000000e+00
Name: customer_previous_avg_delay, dtype: float64


PREVIOUS LATE PAYMENT RATE:


count    1873976.0
mean           0.0
std            0.0
min            0.0
25%            0.0
50%            0.0
75%            0.0
max            0.0
Name: customer_previous_late_rate, dtype: float64


INVOICES WITH NO PREVIOUS PAYMENT HISTORY:
126024

DISTRIBUTION OF PREVIOUS LATE PAYMENT RATE:


customer_previous_late_rate
0.0    1873976
NaN     126024
Name: count, dtype: int64

In [29]:

# ==========================================
# SAMPLE CUSTOMER PAYMENT HISTORY
# ==========================================

display(
    df.loc[
        df["customer_id"] == sample_customer,
        [
            "customer_id",
            "invoice_date",
            "payment_date_clean",
            "payment_delay_days",
            "customer_previous_avg_delay",
            "customer_previous_late_rate"
        ]
    ].sort_values("invoice_date").head(20)
)


,customer_id,invoice_date,payment_date_clean,payment_delay_days,customer_previous_avg_delay,customer_previous_late_rate
915937,CUST-015153,2024-01-05,2024-01-29,-6.0,NaN,NaN
1190331,CUST-015153,2024-01-07,2024-02-01,-20.0,NaN,NaN
1948732,CUST-015153,2024-01-28,2024-02-22,-5.0,NaN,NaN
1439858,CUST-015153,2024-02-09,2024-02-13,-11.0,-13.000000,0.0
1255882,CUST-015153,2024-02-12,NaT,NaN,-13.000000,0.0
1242990,CUST-015153,2024-02-20,NaT,NaN,-12.333333,0.0
850681,CUST-015153,2024-02-24,NaT,NaN,-10.500000,0.0
770928,CUST-015153,2024-03-16,NaT,NaN,-10.500000,0.0
1389009,CUST-015153,2024-03-28,2024-04-14,-28.0,-10.500000,0.0
288044,CUST-015153,2024-03-28,NaT,NaN,-10.500000,0.0


In [30]:

# ==========================================
# TASK B - HISTORY AVAILABILITY INDICATORS
# ==========================================

# Convert previous invoice count to integer
df["customer_previous_invoice_count"] = (
    df["customer_previous_invoice_count"].astype("int32")
)

# Flag customers with no previous invoices
df["customer_no_invoice_history"] = (
    df["customer_previous_invoice_count"] == 0
).astype("int8")

# Flag customers with no previous completed payments
df["customer_no_payment_history"] = (
    df["customer_previous_avg_delay"].isna()
).astype("int8")

print("HISTORY INDICATORS CREATED")

display(
    df[
        [
            "customer_previous_invoice_count",
            "customer_previous_avg_amount",
            "customer_previous_avg_delay",
            "customer_previous_late_rate",
            "customer_no_invoice_history",
            "customer_no_payment_history"
        ]
    ].head(10)
)


HISTORY INDICATORS CREATED


,customer_previous_invoice_count,customer_previous_avg_amount,customer_previous_avg_delay,customer_previous_late_rate,customer_no_invoice_history,customer_no_payment_history
0,45,27618.986444,-15.833333,0.0,0,0
1,21,40522.519524,-12.500000,0.0,0,0
2,25,26420.784800,-16.117647,0.0,0,0
3,27,32878.427037,-15.187500,0.0,0,0
4,7,40926.150000,-6.750000,0.0,0,0
5,12,39771.975000,-14.500000,0.0,0,0
6,31,40477.011290,-18.520000,0.0,0,0
7,33,35599.648182,-14.826087,0.0,0,0
8,34,39345.461471,-21.086957,0.0,0,0
9,20,22162.733500,-12.454545,0.0,0,0


In [31]:

# ==========================================
# TASK C - INVOICE AMOUNT INVESTIGATION
# ==========================================

print("CURRENCY DISTRIBUTION:")
display(df["currency"].value_counts(dropna=False))

print("\nTOTAL AMOUNT SUMMARY:")
display(df["total_amount"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\nTOTAL AMOUNT SKEWNESS:")
print(df["total_amount"].skew())

print("\nTOTAL AMOUNT MISSING VALUES:")
print(df["total_amount"].isna().sum())

print("\nTOTAL AMOUNT ZERO OR NEGATIVE VALUES:")
print((df["total_amount"] <= 0).sum())

print("\nHISTORICAL AMOUNT COMPARISON:")
display(
    df[
        [
            "total_amount",
            "customer_previous_avg_amount"
        ]
    ].head(10)
)


CURRENCY DISTRIBUTION:


currency
INR    600316
USD    598866
EUR    300092
GBP    200817
AUD    160024
CAD    139885
Name: count, dtype: int64


TOTAL AMOUNT SUMMARY:


count    2.000000e+06
mean     3.397916e+04
std      2.989939e+04
min      4.710000e+00
10%      3.118609e+03
25%      9.485370e+03
50%      2.535221e+04
75%      5.149472e+04
90%      7.924087e+04
95%      9.485810e+04
99%      1.193719e+05
max      1.599360e+05
Name: total_amount, dtype: float64


TOTAL AMOUNT SKEWNESS:
1.0307479967862212

TOTAL AMOUNT MISSING VALUES:
0

TOTAL AMOUNT ZERO OR NEGATIVE VALUES:
0

HISTORICAL AMOUNT COMPARISON:


,total_amount,customer_previous_avg_amount
0,66614.93,27618.986444
1,56392.01,40522.519524
2,20879.49,26420.784800
3,102823.20,32878.427037
4,49557.54,40926.150000
5,94697.99,39771.975000
6,101140.92,40477.011290
7,23492.48,35599.648182
8,15209.20,39345.461471
9,13281.83,22162.733500


In [32]:

# ==========================================
# TASK C - CURRENCY CONSISTENCY CHECK
# ==========================================

# Count distinct currencies used by each customer
customer_currency_count = (
    df.groupby("customer_id")["currency"].nunique()
)

print("CUSTOMERS USING MULTIPLE CURRENCIES:")
print((customer_currency_count > 1).sum())

print("\nDISTRIBUTION OF CURRENCIES PER CUSTOMER:")
display(customer_currency_count.value_counts().sort_index())


# Check invoice amount distribution separately for each currency
print("\nAMOUNT DISTRIBUTION BY CURRENCY:")

display(
    df.groupby("currency")["total_amount"].agg(
        ["count", "mean", "median", "min", "max"]
    ).round(2)
)


# Test logarithmic transformation
log_amount = np.log1p(df["total_amount"])

print("\nORIGINAL AMOUNT SKEWNESS:")
print(df["total_amount"].skew())

print("\nLOG-TRANSFORMED AMOUNT SKEWNESS:")
print(log_amount.skew())


CUSTOMERS USING MULTIPLE CURRENCIES:
60000

DISTRIBUTION OF CURRENCIES PER CUSTOMER:


currency
3       26
4      779
5    11070
6    48125
Name: count, dtype: int64


AMOUNT DISTRIBUTION BY CURRENCY:


,count,mean,median,min,max
currency,,,,,
AUD,160024,34001.11,25385.78,7.07,159210.43
CAD,139885,33932.52,25334.03,8.98,159936.00
EUR,300092,33925.16,25199.77,5.84,159692.28
GBP,200817,34102.87,25443.05,5.66,159064.96
INR,600316,33973.63,25386.50,4.71,159850.07
USD,598866,33975.32,25355.97,6.36,159825.76



ORIGINAL AMOUNT SKEWNESS:
1.0307479967862212

LOG-TRANSFORMED AMOUNT SKEWNESS:
-1.0331301349740896


In [33]:

# ==========================================
# TASK C - CURRENCY-AWARE CUSTOMER HISTORY
# ==========================================

# Select the required columns
currency_history = df[
    [
        "customer_id",
        "currency",
        "invoice_date",
        "total_amount"
    ]
].copy()

# Sort customer invoices by currency and date
currency_history = currency_history.sort_values(
    ["customer_id", "currency", "invoice_date"],
    kind="mergesort"
)

# Define grouping levels
customer_currency = [
    "customer_id",
    "currency"
]

customer_currency_date = [
    "customer_id",
    "currency",
    "invoice_date"
]

# Cumulative amount for each customer and currency
cumulative_amount = (
    currency_history.groupby(
        customer_currency,
        sort=False
    )["total_amount"].cumsum()
)

# Cumulative amount within each customer, currency and date
same_day_amount = (
    currency_history.groupby(
        customer_currency_date,
        sort=False
    )["total_amount"].cumsum()
)

# Remove all amounts from the current invoice date
previous_amount_sum = cumulative_amount - same_day_amount

# Count earlier invoices in the same currency
cumulative_count = (
    currency_history.groupby(
        customer_currency,
        sort=False
    ).cumcount()
)

same_day_count = (
    currency_history.groupby(
        customer_currency_date,
        sort=False
    ).cumcount()
)

previous_currency_invoice_count = (
    cumulative_count - same_day_count
)

# Calculate historical average
previous_currency_average = np.divide(
    previous_amount_sum.to_numpy(),
    previous_currency_invoice_count.to_numpy(),
    out=np.full(len(currency_history), np.nan),
    where=previous_currency_invoice_count.to_numpy() > 0
)

# Store the corrected historical average
df.loc[
    currency_history.index,
    "customer_currency_previous_avg_amount"
] = previous_currency_average

# Calculate current amount relative to historical average
df["customer_amount_vs_history"] = (
    df["total_amount"]
    / df["customer_currency_previous_avg_amount"]
)

# Create a flag for customers without previous invoices in that currency
df["customer_no_currency_amount_history"] = (
    df["customer_currency_previous_avg_amount"].isna()
).astype("int8")

print("CURRENCY-AWARE AMOUNT FEATURES CREATED!")


CURRENCY-AWARE AMOUNT FEATURES CREATED!


In [34]:

# ==========================================
# VALIDATE CURRENCY-AWARE AMOUNT FEATURES
# ==========================================

print("MISSING VALUES:")

display(
    df[
        [
            "customer_currency_previous_avg_amount",
            "customer_amount_vs_history"
        ]
    ].isnull().sum()
)

print("\nHISTORICAL AVERAGE AMOUNT SUMMARY:")

display(
    df["customer_currency_previous_avg_amount"].describe()
)

print("\nCURRENT AMOUNT VS HISTORY SUMMARY:")

display(
    df["customer_amount_vs_history"].describe()
)

print("\nINVOICES WITHOUT PREVIOUS SAME-CURRENCY HISTORY:")

print(
    df["customer_no_currency_amount_history"].sum()
)

print("\nSAMPLE CUSTOMER COMPARISON:")

display(
    df.loc[
        df["customer_id"] == sample_customer,
        [
            "customer_id",
            "currency",
            "invoice_date",
            "total_amount",
            "customer_currency_previous_avg_amount",
            "customer_amount_vs_history"
        ]
    ].sort_values(
        ["currency", "invoice_date"]
    ).head(20)
)


MISSING VALUES:


customer_currency_previous_avg_amount    348620
customer_amount_vs_history               348620
dtype: int64


HISTORICAL AVERAGE AMOUNT SUMMARY:


count    1.651380e+06
mean     3.395497e+04
std      1.886596e+04
min      6.430000e+00
25%      2.152234e+04
50%      3.207064e+04
75%      4.342001e+04
max      1.587419e+05
Name: customer_currency_previous_avg_amount, dtype: float64


CURRENT AMOUNT VS HISTORY SUMMARY:


count    1.651380e+06
mean     2.328805e+00
std      2.616086e+01
min      8.618332e-05
25%      2.966340e-01
50%      8.276155e-01
75%      1.847480e+00
max      1.722195e+04
Name: customer_amount_vs_history, dtype: float64


INVOICES WITHOUT PREVIOUS SAME-CURRENCY HISTORY:
348620

SAMPLE CUSTOMER COMPARISON:


,customer_id,currency,invoice_date,total_amount,customer_currency_previous_avg_amount,customer_amount_vs_history
850681,CUST-015153,AUD,2024-02-24,23971.69,NaN,NaN
1983675,CUST-015153,AUD,2024-04-26,65910.30,23971.690000,2.749506
55737,CUST-015153,AUD,2025-02-19,42366.37,44940.995000,0.942711
596851,CUST-015153,AUD,2025-07-04,9819.41,44082.786667,0.222749
1410448,CUST-015153,AUD,2025-08-10,9393.09,35516.942500,0.264468
1392049,CUST-015153,AUD,2025-09-06,44675.23,30292.172000,1.474811
1242990,CUST-015153,CAD,2024-02-20,38767.15,NaN,NaN
584863,CUST-015153,CAD,2024-03-29,21481.94,38767.150000,0.554127
1595458,CUST-015153,CAD,2024-09-03,85814.51,30124.545000,2.848657
1589346,CUST-015153,CAD,2024-12-12,8703.87,48687.866667,0.178769


In [35]:

# ==========================================
# TASK C - FINAL AMOUNT COMPARISON FEATURES
# ==========================================

# Flag invoices without previous same-currency history
df["customer_no_same_currency_history"] = (
    df["customer_currency_previous_avg_amount"].isna()
).astype("int8")


# Calculate logarithmic amount ratio
df["customer_amount_vs_history_log"] = np.log(
    df["customer_amount_vs_history"]
)


print("AMOUNT COMPARISON FEATURES CREATED!")


print("\nMISSING VALUES:")
display(
    df[
        [
            "customer_no_same_currency_history",
            "customer_amount_vs_history_log"
        ]
    ].isnull().sum()
)


print("\nLOG AMOUNT RATIO SUMMARY:")
display(
    df["customer_amount_vs_history_log"].describe()
)


print("\nSAMPLE:")
display(
    df[
        [
            "customer_id",
            "currency",
            "total_amount",
            "customer_currency_previous_avg_amount",
            "customer_amount_vs_history",
            "customer_amount_vs_history_log",
            "customer_no_same_currency_history"
        ]
    ].head(15)
)


AMOUNT COMPARISON FEATURES CREATED!

MISSING VALUES:


customer_no_same_currency_history         0
customer_amount_vs_history_log       348620
dtype: int64


LOG AMOUNT RATIO SUMMARY:


count    1.651380e+06
mean    -3.596469e-01
std      1.498296e+00
min     -9.359034e+00
25%     -1.215256e+00
50%     -1.892066e-01
75%      6.138225e-01
max      9.753940e+00
Name: customer_amount_vs_history_log, dtype: float64


SAMPLE:


,customer_id,currency,total_amount,customer_currency_previous_avg_amount,customer_amount_vs_history,customer_amount_vs_history_log,customer_no_same_currency_history
0,CUST-033946,USD,66614.93,30690.041111,2.170572,0.774991,0
1,CUST-037116,INR,56392.01,55712.530000,1.012196,0.012122,0
2,CUST-027731,GBP,20879.49,26874.926667,0.776913,-0.252426,0
3,CUST-047646,EUR,102823.20,27707.984000,3.710959,1.311290,0
4,CUST-015519,INR,49557.54,80276.630000,0.617335,-0.482344,0
5,CUST-002605,INR,94697.99,46330.226667,2.043979,0.714898,0
6,CUST-051993,USD,101140.92,31809.510000,3.179581,1.156749,0
7,CUST-011891,USD,23492.48,26174.726364,0.897525,-0.108114,0
8,CUST-049600,GBP,15209.20,30795.808571,0.493872,-0.705478,0
9,CUST-028156,INR,13281.83,20767.693333,0.639543,-0.447002,0



## Feature Engineering Catalogue – Tasks A, B and C

### Task A – Time-Based Features

| Feature | Source | Calculation | Data Type | Business Purpose |
|---|---|---|---|---|
| invoice_month | invoice_date | Extract calendar month | Integer | Identify monthly payment patterns |
| invoice_quarter | invoice_date | Extract calendar quarter | Integer | Identify quarterly trends |
| invoice_weekday | invoice_date | Extract weekday (Monday=0) | Integer | Identify weekday-related patterns |
| is_month_end | invoice_date | 1 if invoice falls on the final calendar day of the month, otherwise 0 | Integer | Capture month-end business activity |
| payment_term_days | invoice_date, due_date | due_date - invoice_date | Integer | Measure agreed payment period |
| days_until_due | invoice_date, due_date | due_date - prediction date; prediction date is invoice_date | Integer | Measure time available until payment is due |

### Task B – Customer Behaviour Features

| Feature | Source | Calculation | Data Type | Missing Value Handling |
|---|---|---|---|---|
| customer_previous_invoice_count | customer_id, invoice_date | Count of earlier invoices issued before the current invoice date | Integer | 0 when no earlier invoice exists |
| customer_previous_avg_amount | total_amount, customer_id, invoice_date | Average amount of earlier customer invoices | Float | Missing when no earlier invoice exists |
| customer_previous_avg_delay | payment_delay_days, payment_date_clean | Average delay of payments completed before the current invoice date | Float | Missing when no earlier completed payment exists |
| customer_previous_late_rate | payment_delay_days, payment_date_clean | Proportion of earlier completed payments with delay greater than zero | Float | Missing when no earlier completed payment exists |
| customer_no_invoice_history | customer_previous_invoice_count | 1 when previous invoice count is zero | Integer | Never missing |
| customer_no_payment_history | customer_previous_avg_delay | 1 when no previous completed payment exists | Integer | Never missing |

### Task C – Invoice Amount Features

| Feature | Source | Calculation | Data Type | Missing Value Handling |
|---|---|---|---|---|
| total_amount | total_amount | Original invoice amount | Float | No missing values |
| customer_currency_previous_avg_amount | customer_id, currency, invoice_date, total_amount | Average of earlier invoices for the same customer and currency | Float | Missing when no earlier same-currency invoice exists |
| customer_amount_vs_history | total_amount, customer_currency_previous_avg_amount | Current amount / previous same-currency average | Float | Missing when no historical average exists |
| customer_amount_vs_history_log | customer_amount_vs_history | Natural logarithm of current-to-historical amount ratio | Float | Missing when no historical average exists |
| customer_no_same_currency_history | customer_currency_previous_avg_amount | 1 when no earlier same-currency invoice exists | Integer | Never missing |

### Important Methodology and Assumptions

1. The prediction point is invoice issuance date.
2. Customer history features use strictly earlier invoice or payment dates.
3. Invoices issued on the same date are excluded from each other's invoice history.
4. Payment history includes only completed payments with payment dates strictly before the current invoice date.
5. Customer amount comparisons are calculated separately for each currency because customers use multiple currencies.
6. No currency conversion has been performed.
7. Missing historical values indicate unavailable history and are not automatically replaced with zero.
8. Amount buckets will be created using training data only.
9. The original total_amount feature is retained. A log transformation of total_amount was not selected because it reversed the skewness rather than improving it.
10. Historical late-payment rates are currently zero because the supplied dataset contains no positive payment-delay examples.


In [36]:

# ==========================================
# TASK A, B AND C - FINAL FEATURE CHECK
# ==========================================

engineered_features = [
    "invoice_month",
    "invoice_quarter",
    "invoice_weekday",
    "is_month_end",
    "payment_term_days",
    "days_until_due",
    "customer_previous_invoice_count",
    "customer_previous_avg_amount",
    "customer_previous_avg_delay",
    "customer_previous_late_rate",
    "customer_no_invoice_history",
    "customer_no_payment_history",
    "customer_currency_previous_avg_amount",
    "customer_amount_vs_history",
    "customer_amount_vs_history_log",
    "customer_no_same_currency_history"
]

print("NUMBER OF ENGINEERED FEATURES:")
print(len(engineered_features))

print("\nFEATURE MISSING VALUE SUMMARY:")
display(
    df[engineered_features].isnull().sum()
)

print("\nFEATURE DATA TYPES:")
display(
    df[engineered_features].dtypes
)

print("\nDATASET SHAPE:")
print(df.shape)

print("\nDUPLICATE ROWS:")
print(df.duplicated().sum())


NUMBER OF ENGINEERED FEATURES:
16

FEATURE MISSING VALUE SUMMARY:


invoice_month                                 0
invoice_quarter                               0
invoice_weekday                               0
is_month_end                                  0
payment_term_days                             0
days_until_due                                0
customer_previous_invoice_count               0
customer_previous_avg_amount              61379
customer_previous_avg_delay              126024
customer_previous_late_rate              126024
customer_no_invoice_history                   0
customer_no_payment_history                   0
customer_currency_previous_avg_amount    348620
customer_amount_vs_history               348620
customer_amount_vs_history_log           348620
customer_no_same_currency_history             0
dtype: int64


FEATURE DATA TYPES:


invoice_month                              int32
invoice_quarter                            int32
invoice_weekday                            int32
is_month_end                                int8
payment_term_days                          int64
days_until_due                             int64
customer_previous_invoice_count            int32
customer_previous_avg_amount             float64
customer_previous_avg_delay              float64
customer_previous_late_rate              float64
customer_no_invoice_history                 int8
customer_no_payment_history                 int8
customer_currency_previous_avg_amount    float64
customer_amount_vs_history               float64
customer_amount_vs_history_log           float64
customer_no_same_currency_history           int8
dtype: object


DATASET SHAPE:
(2000000, 56)

DUPLICATE ROWS:
0


In [37]:

# ==========================================
# TASK D - STEP 1: CATEGORICAL FEATURE INSPECTION
# ==========================================

# Identify all categorical columns
categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Create a summary of categorical columns
categorical_summary = pd.DataFrame({
    "Column": categorical_columns,
    "Unique_Values": [
        df[col].nunique(dropna=True)
        for col in categorical_columns
    ],
    "Missing_Values": [
        df[col].isnull().sum()
        for col in categorical_columns
    ]
})

categorical_summary = categorical_summary.sort_values(
    by="Unique_Values",
    ascending=False
)

print("CATEGORICAL COLUMN SUMMARY:")
display(categorical_summary)

print("\nTOTAL CATEGORICAL COLUMNS:")
print(len(categorical_columns))


C:\Users\palar\AppData\Local\Temp\ipykernel_12632\350092549.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


CATEGORICAL COLUMN SUMMARY:


,Column,Unique_Values,Missing_Values
0,invoice_number,2000000,0
1,customer_id,60000,0
3,customer_email,60000,0
2,customer_name,1500,0
16,sales_rep,233,0
9,product_id,46,0
11,product_name,46,0
8,vendor_name,30,0
4,customer_city,30,0
7,vendor_id,30,0



TOTAL CATEGORICAL COLUMNS:
17


In [38]:

# ==========================================
# TASK D - STEP 2: NUMERICAL FEATURE INSPECTION
# ==========================================

numerical_columns = df.select_dtypes(
    include=["int8", "int16", "int32", "int64",
             "float16", "float32", "float64"]
).columns.tolist()

numerical_summary = pd.DataFrame({
    "Column": numerical_columns,
    "Missing_Values": [
        df[col].isnull().sum()
        for col in numerical_columns
    ],
    "Unique_Values": [
        df[col].nunique(dropna=True)
        for col in numerical_columns
    ]
})

print("NUMERICAL COLUMN SUMMARY:")
display(numerical_summary)

print("\nTOTAL NUMERICAL COLUMNS:")
print(len(numerical_columns))


NUMERICAL COLUMN SUMMARY:


,Column,Missing_Values,Unique_Values
0,invoice_id,0,2000000
1,quantity,0,50
2,unit_price,0,249409
3,discount_percent,0,1005
4,tax_percent,0,5
5,subtotal,0,1471475
6,discount_amount,0,645440
7,tax_amount,0,913288
8,shipping_cost,0,15001
9,total_amount,0,1739490



TOTAL NUMERICAL COLUMNS:
35


In [39]:

# ==========================================
# TASK D - STEP 3: CHRONOLOGICAL DATA SPLIT
# ==========================================

# Make sure invoice_date is in datetime format
df["invoice_date"] = pd.to_datetime(
    df["invoice_date"],
    errors="coerce"
)

# Create chronological split labels
df["data_split"] = np.select(
    [
        df["invoice_date"] < "2025-01-01",

        (df["invoice_date"] >= "2025-01-01") &
        (df["invoice_date"] < "2025-07-01"),

        (df["invoice_date"] >= "2025-07-01") &
        (df["invoice_date"] <= "2025-12-31")
    ],
    [
        "train",
        "validation",
        "test"
    ],
    default="unassigned"
)

# Display split counts
print("CHRONOLOGICAL DATA SPLIT:")
display(
    df["data_split"].value_counts().to_frame(
        name="Number_of_Invoices"
    )
)

# Display split percentages
print("\nSPLIT PERCENTAGES:")
display(
    (
        df["data_split"].value_counts(normalize=True) * 100
    ).round(2).to_frame(
        name="Percentage"
    )
)

# Check date boundaries
print("\nDATE RANGE BY SPLIT:")
display(
    df.groupby("data_split")["invoice_date"].agg(
        ["min", "max", "count"]
    )
)

# Check unassigned records
print("\nUNASSIGNED RECORDS:")
print((df["data_split"] == "unassigned").sum())


CHRONOLOGICAL DATA SPLIT:


,Number_of_Invoices
data_split,
train,1001116
test,503537
validation,495347



SPLIT PERCENTAGES:


,Percentage
data_split,
train,50.06
test,25.18
validation,24.77



DATE RANGE BY SPLIT:


,min,max,count
data_split,,,
test,2025-07-01,2025-12-31,503537
train,2024-01-01,2024-12-31,1001116
validation,2025-01-01,2025-06-30,495347



UNASSIGNED RECORDS:
0


In [40]:

# ==========================================
# TASK D - STEP 4: TARGET AVAILABILITY CHECK
# ==========================================

# A valid payment outcome requires:
# 1. Payment status is Paid
# 2. Actual cleaned payment date is available
# 3. Payment delay is available

valid_paid_outcome = (
    df["payment_status"].eq("Paid") &
    df["payment_date_clean"].notna() &
    df["payment_delay_days"].notna()
)

# Create a summary for each split
target_summary = []

for split_name in ["train", "validation", "test"]:

    split_df = df[df["data_split"] == split_name]

    valid_delays = split_df.loc[
        valid_paid_outcome.loc[split_df.index],
        "payment_delay_days"
    ]

    target_summary.append({
        "Dataset": split_name,
        "Total_Invoices": len(split_df),
        "Valid_Paid_Outcomes": len(valid_delays),
        "Missing_Payment_Outcomes": len(split_df) - len(valid_delays),
        "Early_Payments": (valid_delays < 0).sum(),
        "On_Time_Payments": (valid_delays == 0).sum(),
        "Late_Payments": (valid_delays > 0).sum()
    })

target_summary_df = pd.DataFrame(target_summary)

print("TARGET AVAILABILITY BY DATA SPLIT:")
display(target_summary_df)

# Overall classification target check
print("\nOVERALL CLASSIFICATION TARGET DISTRIBUTION:")

paid_delays = df.loc[
    valid_paid_outcome,
    "payment_delay_days"
]

print("Not late (delay <= 0):", (paid_delays <= 0).sum())
print("Late (delay > 0):", (paid_delays > 0).sum())

# Verify target eligibility
print("\nVALID PAID OUTCOME RECORDS:")
print(valid_paid_outcome.sum())

print("\nTOTAL RECORDS:")
print(len(df))


TARGET AVAILABILITY BY DATA SPLIT:


,Dataset,Total_Invoices,Valid_Paid_Outcomes,Missing_Payment_Outcomes,Early_Payments,On_Time_Payments,Late_Payments
0,train,1001116,680398,320718,653476,26922,0
1,validation,495347,336301,159046,322994,13307,0
2,test,503537,342120,161417,328597,13523,0



OVERALL CLASSIFICATION TARGET DISTRIBUTION:
Not late (delay <= 0): 1358819
Late (delay > 0): 0

VALID PAID OUTCOME RECORDS:
1358819

TOTAL RECORDS:
2000000


In [ ]:

# ==========================================
# PAYMENT STATUS ANALYSIS
# ==========================================

print("PAYMENT STATUS DISTRIBUTION:")
display(df["payment_status"].value_counts(dropna=False))

print("\nPAID FLAG DISTRIBUTION:")
display(df["is_paid"].value_counts(dropna=False))

print("\nOPEN FLAG DISTRIBUTION:")
display(df["is_open"].value_counts(dropna=False))

print("\nPAID LATE FLAG DISTRIBUTION:")
display(df["paid_late_flag"].value_counts(dropna=False))

print("\nOVERDUE TARGET DISTRIBUTION:")
display(df["overdue_target"].value_counts(dropna=False))


PAYMENT STATUS DISTRIBUTION:


payment_status
Paid         1358819
Pending       341015
Overdue       199777
Cancelled     100389
Name: count, dtype: int64


PAID FLAG DISTRIBUTION:


is_paid
1    1358819
0     641181
Name: count, dtype: int64


OPEN FLAG DISTRIBUTION:


is_open
0    1459208
1     540792
Name: count, dtype: int64


PAID LATE FLAG DISTRIBUTION:


paid_late_flag
0.0    1358819
NaN     641181
Name: count, dtype: int64


OVERDUE TARGET DISTRIBUTION:


overdue_target
0    1800223
1     199777
Name: count, dtype: int64

In [41]:
print("CURRENCY HISTORY INDICATOR CHECK:")

for col in [
    "customer_no_currency_amount_history",
    "customer_no_same_currency_history"
]:
    print(
        col,
        "Exists:", col in df.columns,
        "Missing:", df[col].isnull().sum() if col in df.columns else "N/A"
    )

CURRENCY HISTORY INDICATOR CHECK:
customer_no_currency_amount_history Exists: True Missing: 0
customer_no_same_currency_history Exists: True Missing: 0


In [42]:

# Check whether the two currency-history indicators are identical

print(
    "Are both currency-history indicators identical?",
    (
        df["customer_no_currency_amount_history"]
        == df["customer_no_same_currency_history"]
    ).all()
)

# Target availability by chronological split

valid_paid = (
    df["payment_status"].eq("Paid")
    & df["payment_date_clean"].notna()
    & df["payment_delay_days"].notna()
)

rows = []

for split in ["train", "validation", "test"]:

    part = df[df["data_split"] == split]

    delays = part.loc[
        valid_paid.loc[part.index],
        "payment_delay_days"
    ]

    rows.append({
        "Split": split,
        "Total_Invoices": len(part),
        "Valid_Paid_Outcomes": len(delays),
        "Missing_Outcomes": len(part) - len(delays),
        "Early": (delays < 0).sum(),
        "On_Time": (delays == 0).sum(),
        "Late": (delays > 0).sum()
    })

display(pd.DataFrame(rows))


Are both currency-history indicators identical? True


,Split,Total_Invoices,Valid_Paid_Outcomes,Missing_Outcomes,Early,On_Time,Late
0,train,1001116,680398,320718,653476,26922,0
1,validation,495347,336301,159046,322994,13307,0
2,test,503537,342120,161417,328597,13523,0


In [43]:

# ==========================================
# TASK D - STEP 5: REMOVE DUPLICATE FEATURE
# ==========================================

# Keep customer_no_same_currency_history
# Remove its identical duplicate

df.drop(
    columns=["customer_no_currency_amount_history"],
    inplace=True
)

print("Duplicate currency-history indicator removed.")
print("Remaining indicator:")
print("customer_no_same_currency_history" in df.columns)


Duplicate currency-history indicator removed.
Remaining indicator:
True


C:\Users\palar\AppData\Local\Temp\ipykernel_12632\2480621378.py:8: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(


In [44]:

# ==========================================
# TASK D - STEP 6: CREATE MODEL TARGETS
# ==========================================

# A valid target requires a completed paid invoice
valid_paid = (
    df["payment_status"].eq("Paid")
    & df["payment_date_clean"].notna()
    & df["payment_delay_days"].notna()
)

# Regression target
df["delay_days"] = np.nan

df.loc[valid_paid, "delay_days"] = (
    df.loc[valid_paid, "payment_delay_days"]
)

# Classification target
# Use pandas nullable integer type so unknown outcomes remain <NA>
df["paid_late"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="Int8"
)

df.loc[valid_paid, "paid_late"] = (
    df.loc[valid_paid, "payment_delay_days"] > 0
).astype("int8")

# Display target distributions
print("REGRESSION TARGET: delay_days")
display(df["delay_days"].describe())

print("\nCLASSIFICATION TARGET: paid_late")
display(df["paid_late"].value_counts(dropna=False))

# Check late-payment examples
print("\nNUMBER OF LATE-PAYMENT EXAMPLES:")
print((df["paid_late"] == 1).sum())

print("\nNUMBER OF VALID REGRESSION TARGETS:")
print(df["delay_days"].notna().sum())

print("\nNUMBER OF UNKNOWN PAYMENT OUTCOMES:")
print(df["delay_days"].isna().sum())


REGRESSION TARGET: delay_days


count    1.358819e+06
mean    -1.501869e+01
std      1.216216e+01
min     -6.000000e+01
25%     -2.200000e+01
50%     -1.200000e+01
75%     -6.000000e+00
max      0.000000e+00
Name: delay_days, dtype: float64


CLASSIFICATION TARGET: paid_late


paid_late
0       1358819
<NA>     641181
Name: count, dtype: Int64


NUMBER OF LATE-PAYMENT EXAMPLES:
0

NUMBER OF VALID REGRESSION TARGETS:
1358819

NUMBER OF UNKNOWN PAYMENT OUTCOMES:
641181


In [45]:

# ==========================================
# TASK D - STEP 7: FEATURE REDUNDANCY CHECK
# ==========================================

# Check whether payment-term columns are identical
print("PAYMENT TERM CHECK:")

print(
    "payment_term_days == days_until_due:",
    df["payment_term_days"].equals(df["days_until_due"])
)

print(
    "payment_term_days == invoice_to_due_days:",
    df["payment_term_days"].equals(df["invoice_to_due_days"])
)

# Check whether product ID maps consistently to product name
print("\nPRODUCT MAPPING CHECK:")

print(
    "Maximum product names per product ID:",
    df.groupby("product_id")["product_name"].nunique().max()
)

print(
    "Maximum product IDs per product name:",
    df.groupby("product_name")["product_id"].nunique().max()
)

# Check whether vendor ID maps consistently to vendor name
print("\nVENDOR MAPPING CHECK:")

print(
    "Maximum vendor names per vendor ID:",
    df.groupby("vendor_id")["vendor_name"].nunique().max()
)

print(
    "Maximum vendor IDs per vendor name:",
    df.groupby("vendor_name")["vendor_id"].nunique().max()
)

# Check historical late-rate variation
print("\nHISTORICAL LATE RATE CHECK:")

display(
    df["customer_previous_late_rate"].value_counts(
        dropna=False
    )
)

# Check final dataset columns
print("\nCURRENT DATASET SHAPE:")
print(df.shape)


PAYMENT TERM CHECK:
payment_term_days == days_until_due: True
payment_term_days == invoice_to_due_days: True

PRODUCT MAPPING CHECK:
Maximum product names per product ID: 1
Maximum product IDs per product name: 1

VENDOR MAPPING CHECK:
Maximum vendor names per vendor ID: 1
Maximum vendor IDs per vendor name: 1

HISTORICAL LATE RATE CHECK:


customer_previous_late_rate
0.0    1873976
NaN     126024
Name: count, dtype: int64


CURRENT DATASET SHAPE:
(2000000, 58)


In [46]:

# ==========================================
# TASK D - STEP 8: FINAL MODEL FEATURE LISTS
# ==========================================

# 1. Numerical features
numeric_features = [
    "quantity",
    "unit_price",
    "discount_percent",
    "tax_percent",
    "subtotal",
    "discount_amount",
    "tax_amount",
    "shipping_cost",
    "total_amount",

    # Time features
    "invoice_month",
    "invoice_weekday",
    "is_month_end",
    "payment_term_days",

    # Customer history features
    "customer_previous_invoice_count",
    "customer_previous_avg_delay",
    "customer_currency_previous_avg_amount",

    # Amount behaviour
    "customer_amount_vs_history_log",

    # History indicators
    "customer_no_invoice_history",
    "customer_no_payment_history",
    "customer_no_same_currency_history"
]


# 2. Categorical features for One-Hot Encoding
onehot_features = [
    "currency",
    "payment_method",
    "order_channel",
    "product_category",
    "product_name",
    "vendor_id",
    "customer_country",
    "customer_state",
    "customer_city"
]


# 3. High-cardinality feature for Frequency Encoding
frequency_features = [
    "sales_rep"
]


# 4. Target column for regression
target_column = "delay_days"


# 5. Columns retained separately for traceability
metadata_columns = [
    "invoice_id",
    "invoice_number",
    "customer_id",
    "invoice_date",
    "data_split"
]


# ==========================================
# VALIDATION
# ==========================================

all_predictor_features = (
    numeric_features
    + onehot_features
    + frequency_features
)

print("NUMERICAL FEATURES:", len(numeric_features))
print("ONE-HOT FEATURES:", len(onehot_features))
print("FREQUENCY FEATURES:", len(frequency_features))
print("TOTAL PREDICTOR FEATURES:", len(all_predictor_features))

# Check whether any selected feature is missing
missing_columns = [
    col for col in all_predictor_features
    if col not in df.columns
]

print("\nMISSING FEATURE COLUMNS:")
print(missing_columns)

# Check duplicate feature names
print("\nDUPLICATE FEATURE NAMES:")
print(
    pd.Series(all_predictor_features)
    .duplicated()
    .sum()
)

# Check that target and leakage columns are excluded
leakage_columns = [
    "payment_status",
    "payment_date",
    "payment_date_clean",
    "days_to_pay",
    "payment_delay_days",
    "paid_late_flag",
    "paid_late",
    "is_paid",
    "is_open",
    "is_overdue",
    "overdue_target"
]

leakage_in_features = [
    col for col in all_predictor_features
    if col in leakage_columns
]

print("\nLEAKAGE COLUMNS IN PREDICTORS:")
print(leakage_in_features)

# Display selected features
print("\nFINAL NUMERICAL FEATURES:")
print(numeric_features)

print("\nFINAL ONE-HOT FEATURES:")
print(onehot_features)

print("\nFINAL FREQUENCY FEATURES:")
print(frequency_features)


NUMERICAL FEATURES: 20
ONE-HOT FEATURES: 9
FREQUENCY FEATURES: 1
TOTAL PREDICTOR FEATURES: 30

MISSING FEATURE COLUMNS:
[]

DUPLICATE FEATURE NAMES:
0

LEAKAGE COLUMNS IN PREDICTORS:
[]

FINAL NUMERICAL FEATURES:
['quantity', 'unit_price', 'discount_percent', 'tax_percent', 'subtotal', 'discount_amount', 'tax_amount', 'shipping_cost', 'total_amount', 'invoice_month', 'invoice_weekday', 'is_month_end', 'payment_term_days', 'customer_previous_invoice_count', 'customer_previous_avg_delay', 'customer_currency_previous_avg_amount', 'customer_amount_vs_history_log', 'customer_no_invoice_history', 'customer_no_payment_history', 'customer_no_same_currency_history']

FINAL ONE-HOT FEATURES:
['currency', 'payment_method', 'order_channel', 'product_category', 'product_name', 'vendor_id', 'customer_country', 'customer_state', 'customer_city']

FINAL FREQUENCY FEATURES:
['sales_rep']


In [47]:
import sklearn

print("Scikit-learn version:", sklearn.__version__)

Scikit-learn version: 1.9.1


In [48]:

# ==========================================
# TASK D - STEP 9: PREPARE LABELLED DATASETS
# ==========================================

# Identify invoices with known regression targets
valid_target = df["delay_days"].notna()

# Create masks for each chronological split
train_mask = (
    (df["data_split"] == "train") &
    valid_target
)

validation_mask = (
    (df["data_split"] == "validation") &
    valid_target
)

test_mask = (
    (df["data_split"] == "test") &
    valid_target
)

# Separate predictor features (X)
X_train_raw = df.loc[
    train_mask, all_predictor_features
].copy()

X_validation_raw = df.loc[
    validation_mask, all_predictor_features
].copy()

X_test_raw = df.loc[
    test_mask, all_predictor_features
].copy()

# Separate regression targets (y)
y_train = df.loc[train_mask, "delay_days"].copy()

y_validation = df.loc[
    validation_mask, "delay_days"
].copy()

y_test = df.loc[test_mask, "delay_days"].copy()

# Preserve metadata separately
metadata_train = df.loc[
    train_mask, metadata_columns
].copy()

metadata_validation = df.loc[
    validation_mask, metadata_columns
].copy()

metadata_test = df.loc[
    test_mask, metadata_columns
].copy()


# ==========================================
# VALIDATION
# ==========================================

print("LABELLED DATASET SHAPES:")

print("\nTraining:")
print("X:", X_train_raw.shape)
print("y:", y_train.shape)

print("\nValidation:")
print("X:", X_validation_raw.shape)
print("y:", y_validation.shape)

print("\nTesting:")
print("X:", X_test_raw.shape)
print("y:", y_test.shape)


print("\nTARGET DISTRIBUTION:")

display(
    pd.DataFrame({
        "Training": y_train.describe(),
        "Validation": y_validation.describe(),
        "Testing": y_test.describe()
    })
)


# Check for missing targets
print("\nMISSING TARGET COUNTS:")

print("Training:", y_train.isna().sum())
print("Validation:", y_validation.isna().sum())
print("Testing:", y_test.isna().sum())


# Check frequency-encoding category counts
print("\nSALES REPRESENTATIVE CATEGORY COUNTS:")

print(
    "Training unique sales reps:",
    X_train_raw["sales_rep"].nunique()
)

print(
    "Validation unique sales reps:",
    X_validation_raw["sales_rep"].nunique()
)

print(
    "Testing unique sales reps:",
    X_test_raw["sales_rep"].nunique()
)


LABELLED DATASET SHAPES:

Training:
X: (680398, 30)
y: (680398,)

Validation:
X: (336301, 30)
y: (336301,)

Testing:
X: (342120, 30)
y: (342120,)

TARGET DISTRIBUTION:


,Training,Validation,Testing
count,680398.000000,336301.000000,342120.000000
mean,-15.005231,-15.041076,-15.023468
std,12.163772,12.155756,12.165244
min,-60.000000,-60.000000,-60.000000
25%,-22.000000,-22.000000,-22.000000
50%,-12.000000,-12.000000,-12.000000
75%,-6.000000,-6.000000,-6.000000
max,0.000000,0.000000,0.000000



MISSING TARGET COUNTS:
Training: 0
Validation: 0
Testing: 0

SALES REPRESENTATIVE CATEGORY COUNTS:
Training unique sales reps: 233
Validation unique sales reps: 233
Testing unique sales reps: 233


In [49]:

# ==========================================
# TASK D - STEP 9: IMPORT PREPROCESSING TOOLS
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from scipy.sparse import csr_matrix, hstack, issparse

print("Preprocessing libraries imported successfully.")


Preprocessing libraries imported successfully.


In [51]:
# Display all currently defined dataset variables

for name in sorted(globals()):
    if name.startswith(("X_", "y_")):
        value = globals()[name]

        if isinstance(value, (pd.DataFrame, pd.Series)):
            print(name, "->", value.shape)

X_test_raw -> (342120, 30)
X_train_raw -> (680398, 30)
X_validation_raw -> (336301, 30)
y_test -> (342120,)
y_train -> (680398,)
y_validation -> (336301,)


In [52]:

# Create frequency mapping using training data only
sales_rep_frequency_map = (
    X_train_raw["sales_rep"]
    .value_counts(normalize=True)
    .to_dict()
)

# Apply the training frequency mapping to all three datasets
X_train_frequency = (
    X_train_raw["sales_rep"]
    .map(sales_rep_frequency_map)
    .fillna(0)
)

X_validation_frequency = (
    X_validation_raw["sales_rep"]
    .map(sales_rep_frequency_map)
    .fillna(0)
)

X_test_frequency = (
    X_test_raw["sales_rep"]
    .map(sales_rep_frequency_map)
    .fillna(0)
)

# Display verification
print("TRAIN FREQUENCY SHAPE:", X_train_frequency.shape)
print("VALIDATION FREQUENCY SHAPE:", X_validation_frequency.shape)
print("TEST FREQUENCY SHAPE:", X_test_frequency.shape)

print("\nMISSING VALUES:")
print("Train:", X_train_frequency.isna().sum())
print("Validation:", X_validation_frequency.isna().sum())
print("Test:", X_test_frequency.isna().sum())

print("\nSAMPLE FREQUENCY VALUES:")
print(X_train_frequency.head())


TRAIN FREQUENCY SHAPE: (680398,)
VALIDATION FREQUENCY SHAPE: (336301,)
TEST FREQUENCY SHAPE: (342120,)

MISSING VALUES:
Train: 0
Validation: 0
Test: 0

SAMPLE FREQUENCY VALUES:
16    0.003933
17    0.004102
18    0.004089
20    0.003961
23    0.008228
Name: sales_rep, dtype: float64


In [53]:

# Import preprocessing tools
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Import sparse matrix tools
from scipy.sparse import csr_matrix, hstack, issparse

# Numerical preprocessing pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

# Combine both pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, onehot_features)
    ],
    remainder="drop",
    sparse_threshold=1.0
)

# Remove original sales_rep column.
# Its frequency-encoded version will be added separately.
X_train_base = X_train_raw.drop(columns=frequency_features)
X_validation_base = X_validation_raw.drop(columns=frequency_features)
X_test_base = X_test_raw.drop(columns=frequency_features)

# Verify feature counts
print("TRAIN BASE SHAPE:", X_train_base.shape)
print("VALIDATION BASE SHAPE:", X_validation_base.shape)
print("TEST BASE SHAPE:", X_test_base.shape)

print("\nEXPECTED NUMERICAL FEATURES:", len(numeric_features))
print("EXPECTED ONE-HOT FEATURES:", len(onehot_features))
print("EXPECTED FREQUENCY FEATURES:", len(frequency_features))

print("\nPREPROCESSOR CREATED SUCCESSFULLY!")


TRAIN BASE SHAPE: (680398, 29)
VALIDATION BASE SHAPE: (336301, 29)
TEST BASE SHAPE: (342120, 29)

EXPECTED NUMERICAL FEATURES: 20
EXPECTED ONE-HOT FEATURES: 9
EXPECTED FREQUENCY FEATURES: 1

PREPROCESSOR CREATED SUCCESSFULLY!


In [54]:

# Fit preprocessing on training data and transform it
X_train_encoded_base = preprocessor.fit_transform(X_train_base)

# Transform validation and test data using the same fitted preprocessor
X_validation_encoded_base = preprocessor.transform(X_validation_base)

X_test_encoded_base = preprocessor.transform(X_test_base)

# Check output shapes
print("TRAIN ENCODED SHAPE:", X_train_encoded_base.shape)
print("VALIDATION ENCODED SHAPE:", X_validation_encoded_base.shape)
print("TEST ENCODED SHAPE:", X_test_encoded_base.shape)

# Check whether outputs are sparse matrices
print("\nSPARSE MATRIX CHECK:")
print("Train:", issparse(X_train_encoded_base))
print("Validation:", issparse(X_validation_encoded_base))
print("Test:", issparse(X_test_encoded_base))

# Check fitted preprocessing
print("\nPREPROCESSOR FITTED SUCCESSFULLY!")


TRAIN ENCODED SHAPE: (680398, 188)
VALIDATION ENCODED SHAPE: (336301, 188)
TEST ENCODED SHAPE: (342120, 188)

SPARSE MATRIX CHECK:
Train: True
Validation: True
Test: True

PREPROCESSOR FITTED SUCCESSFULLY!


In [55]:

# Convert frequency-encoded Series into sparse matrices
X_train_frequency_matrix = csr_matrix(
    X_train_frequency.to_numpy().reshape(-1, 1)
)

X_validation_frequency_matrix = csr_matrix(
    X_validation_frequency.to_numpy().reshape(-1, 1)
)

X_test_frequency_matrix = csr_matrix(
    X_test_frequency.to_numpy().reshape(-1, 1)
)

# Combine preprocessed features with frequency encoding
X_train_preprocessed = hstack([
    X_train_encoded_base,
    X_train_frequency_matrix
]).tocsr()

X_validation_preprocessed = hstack([
    X_validation_encoded_base,
    X_validation_frequency_matrix
]).tocsr()

X_test_preprocessed = hstack([
    X_test_encoded_base,
    X_test_frequency_matrix
]).tocsr()

# Display final shapes
print("FINAL TRAIN SHAPE:", X_train_preprocessed.shape)
print("FINAL VALIDATION SHAPE:", X_validation_preprocessed.shape)
print("FINAL TEST SHAPE:", X_test_preprocessed.shape)

# Verify sparse matrix format
print("\nSPARSE MATRIX CHECK:")
print("Train:", issparse(X_train_preprocessed))
print("Validation:", issparse(X_validation_preprocessed))
print("Test:", issparse(X_test_preprocessed))

# Verify feature count
print("\nTOTAL FINAL FEATURES:", X_train_preprocessed.shape[1])



FINAL TRAIN SHAPE: (680398, 189)
FINAL VALIDATION SHAPE: (336301, 189)
FINAL TEST SHAPE: (342120, 189)

SPARSE MATRIX CHECK:
Train: True
Validation: True
Test: True

TOTAL FINAL FEATURES: 189


In [56]:

import numpy as np
import pandas as pd

# Get names of numerical and one-hot encoded features
encoded_feature_names = preprocessor.get_feature_names_out()

# Add the frequency-encoded feature name
final_feature_names = list(encoded_feature_names) + [
    "sales_rep_frequency"
]

# Convert into a DataFrame for easy inspection
feature_catalogue = pd.DataFrame({
    "Feature_Number": range(1, len(final_feature_names) + 1),
    "Feature_Name": final_feature_names
})

# Check feature names
print("TOTAL FEATURE NAMES:", len(final_feature_names))
print("UNIQUE FEATURE NAMES:", len(set(final_feature_names)))

print("\nFIRST 20 FEATURES:")
print(feature_catalogue.head(20).to_string(index=False))

print("\nLAST 10 FEATURES:")
print(feature_catalogue.tail(10).to_string(index=False))

# Verify target alignment
print("\nTARGET ALIGNMENT:")
print("Train:", X_train_preprocessed.shape[0], len(y_train))
print("Validation:", X_validation_preprocessed.shape[0], len(y_validation))
print("Test:", X_test_preprocessed.shape[0], len(y_test))

# Check for invalid numerical values in sparse matrices
print("\nINVALID VALUE CHECK:")
print("Train:", np.isfinite(X_train_preprocessed.data).all())
print("Validation:", np.isfinite(X_validation_preprocessed.data).all())
print("Test:", np.isfinite(X_test_preprocessed.data).all())

# Final verification
print("\nFINAL FEATURE COUNT MATCH:")
print(
    X_train_preprocessed.shape[1] == len(final_feature_names)
)


TOTAL FEATURE NAMES: 189
UNIQUE FEATURE NAMES: 189

FIRST 20 FEATURES:
 Feature_Number                                   Feature_Name
              1                              numeric__quantity
              2                            numeric__unit_price
              3                      numeric__discount_percent
              4                           numeric__tax_percent
              5                              numeric__subtotal
              6                       numeric__discount_amount
              7                            numeric__tax_amount
              8                         numeric__shipping_cost
              9                          numeric__total_amount
             10                         numeric__invoice_month
             11                       numeric__invoice_weekday
             12                          numeric__is_month_end
             13                     numeric__payment_term_days
             14       numeric__customer_previou

In [57]:
import os

print("CURRENT WORKING DIRECTORY:")
print(os.getcwd())

print("\nPROJECT FOLDERS:")
print(os.listdir())

CURRENT WORKING DIRECTORY:
c:\Users\palar\OneDrive\Desktop\InvoiceIQ

PROJECT FOLDERS:
['.git', '.gitignore', '.venv', 'data', 'notebooks', 'README.md', 'reports', 'requirements.txt', 'scripts']


In [58]:

import os
import numpy as np
import joblib
from scipy.sparse import save_npz

# 1. Create processed data folder
processed_folder = os.path.join("data", "processed")
os.makedirs(processed_folder, exist_ok=True)

# 2. Save sparse feature matrices
save_npz(
    os.path.join(processed_folder, "X_train.npz"),
    X_train_preprocessed
)

save_npz(
    os.path.join(processed_folder, "X_validation.npz"),
    X_validation_preprocessed
)

save_npz(
    os.path.join(processed_folder, "X_test.npz"),
    X_test_preprocessed
)

# 3. Save target values
np.save(
    os.path.join(processed_folder, "y_train.npy"),
    y_train.to_numpy()
)

np.save(
    os.path.join(processed_folder, "y_validation.npy"),
    y_validation.to_numpy()
)

np.save(
    os.path.join(processed_folder, "y_test.npy"),
    y_test.to_numpy()
)

# 4. Save feature catalogue
feature_catalogue.to_csv(
    os.path.join(processed_folder, "feature_catalogue.csv"),
    index=False
)

# 5. Save fitted preprocessing objects
joblib.dump(
    preprocessor,
    os.path.join(processed_folder, "preprocessor.joblib")
)

joblib.dump(
    sales_rep_frequency_map,
    os.path.join(processed_folder, "sales_rep_frequency_map.joblib")
)

# 6. Display saved files
print("FILES SAVED SUCCESSFULLY!\n")

for filename in sorted(os.listdir(processed_folder)):
    filepath = os.path.join(processed_folder, filename)
    print(filename, "->", round(os.path.getsize(filepath) / (1024 ** 2), 2), "MB")

print("\nTOTAL FILES:", len(os.listdir(processed_folder)))


FILES SAVED SUCCESSFULLY!

X_test.npz -> 29.84 MB
X_train.npz -> 55.42 MB
X_validation.npz -> 29.08 MB
feature_catalogue.csv -> 0.01 MB
preprocessor.joblib -> 0.01 MB
sales_rep_frequency_map.joblib -> 0.01 MB
y_test.npy -> 2.61 MB
y_train.npy -> 5.19 MB
y_validation.npy -> 2.57 MB

TOTAL FILES: 9


In [59]:

import os
import numpy as np
import joblib
import pandas as pd
from scipy.sparse import load_npz

# Folder containing saved files
processed_folder = os.path.join("data", "processed")

# 1. Load feature matrices
X_train_loaded = load_npz(
    os.path.join(processed_folder, "X_train.npz")
)

X_validation_loaded = load_npz(
    os.path.join(processed_folder, "X_validation.npz")
)

X_test_loaded = load_npz(
    os.path.join(processed_folder, "X_test.npz")
)

# 2. Load target arrays
y_train_loaded = np.load(
    os.path.join(processed_folder, "y_train.npy")
)

y_validation_loaded = np.load(
    os.path.join(processed_folder, "y_validation.npy")
)

y_test_loaded = np.load(
    os.path.join(processed_folder, "y_test.npy")
)

# 3. Load feature catalogue
feature_catalogue_loaded = pd.read_csv(
    os.path.join(processed_folder, "feature_catalogue.csv")
)

# 4. Load fitted preprocessing objects
preprocessor_loaded = joblib.load(
    os.path.join(processed_folder, "preprocessor.joblib")
)

frequency_map_loaded = joblib.load(
    os.path.join(processed_folder, "sales_rep_frequency_map.joblib")
)

# 5. Verify dimensions
print("LOADED DATASET SHAPES:")
print("Train:", X_train_loaded.shape, y_train_loaded.shape)
print("Validation:", X_validation_loaded.shape, y_validation_loaded.shape)
print("Test:", X_test_loaded.shape, y_test_loaded.shape)

print("\nFEATURE CATALOGUE SHAPE:")
print(feature_catalogue_loaded.shape)

print("\nPREPROCESSOR LOADED:", preprocessor_loaded is not None)
print("FREQUENCY MAP ENTRIES:", len(frequency_map_loaded))

# 6. Verify data consistency
print("\nDATA CONSISTENCY CHECK:")
print(
    "Training features match:",
    (X_train_loaded != X_train_preprocessed).nnz == 0
)

print(
    "Validation features match:",
    (X_validation_loaded != X_validation_preprocessed).nnz == 0
)

print(
    "Testing features match:",
    (X_test_loaded != X_test_preprocessed).nnz == 0
)

print(
    "Training targets match:",
    np.array_equal(y_train_loaded, y_train.to_numpy(), equal_nan=True)
)

print(
    "Validation targets match:",
    np.array_equal(y_validation_loaded, y_validation.to_numpy(), equal_nan=True)
)

print(
    "Testing targets match:",
    np.array_equal(y_test_loaded, y_test.to_numpy(), equal_nan=True)
)


LOADED DATASET SHAPES:
Train: (680398, 189) (680398,)
Validation: (336301, 189) (336301,)
Test: (342120, 189) (342120,)

FEATURE CATALOGUE SHAPE:
(189, 2)

PREPROCESSOR LOADED: True
FREQUENCY MAP ENTRIES: 233

DATA CONSISTENCY CHECK:
Training features match: True
Validation features match: True
Testing features match: True
Training targets match: True
Validation targets match: True
Testing targets match: True


In [60]:

# Create chronological split summary

split_summary = (
    df.groupby("data_split")
    .agg(
        Total_Records=("invoice_id", "count"),
        Start_Date=("invoice_date", "min"),
        End_Date=("invoice_date", "max"),
        Available_Regression_Targets=("delay_days", "count"),
        Unknown_Payment_Outcomes=("delay_days", lambda x: x.isna().sum()),
        Early_Payments=("delay_days", lambda x: (x < 0).sum()),
        On_Time_Payments=("delay_days", lambda x: (x == 0).sum()),
        Late_Payments=("delay_days", lambda x: (x > 0).sum())
    )
    .reset_index()
)

# Calculate percentage of total records
split_summary["Percentage_of_Dataset"] = (
    split_summary["Total_Records"] / len(df) * 100
).round(2)

# Save summary
split_summary.to_csv(
    os.path.join(processed_folder, "chronological_split_summary.csv"),
    index=False
)

# Display results
print("CHRONOLOGICAL SPLIT SUMMARY:\n")
print(split_summary.to_string(index=False))

print("\nSPLIT SUMMARY SAVED SUCCESSFULLY!")


CHRONOLOGICAL SPLIT SUMMARY:

data_split  Total_Records Start_Date   End_Date  Available_Regression_Targets  Unknown_Payment_Outcomes  Early_Payments  On_Time_Payments  Late_Payments  Percentage_of_Dataset
      test         503537 2025-07-01 2025-12-31                        342120                    161417          328597             13523              0                  25.18
     train        1001116 2024-01-01 2024-12-31                        680398                    320718          653476             26922              0                  50.06
validation         495347 2025-01-01 2025-06-30                        336301                    159046          322994             13307              0                  24.77

SPLIT SUMMARY SAVED SUCCESSFULLY!


In [61]:

# Create detailed feature catalogue

feature_details = []

for feature in final_feature_names:

    if feature.startswith("numeric__"):
        original_name = feature.replace("numeric__", "")
        feature_group = "Numerical"
        data_type = "Numeric"

        descriptions = {
            "quantity": "Number of units purchased",
            "unit_price": "Price per unit",
            "discount_percent": "Discount percentage applied",
            "tax_percent": "Tax percentage applied",
            "subtotal": "Invoice subtotal before additional charges",
            "discount_amount": "Total discount amount",
            "tax_amount": "Tax amount",
            "shipping_cost": "Shipping cost",
            "total_amount": "Final invoice amount",
            "invoice_month": "Month in which invoice was generated",
            "invoice_weekday": "Day of week when invoice was generated",
            "is_month_end": "Indicates whether invoice date is month end",
            "payment_term_days": "Number of days between invoice date and due date",
            "customer_previous_invoice_count": "Number of earlier customer invoices",
            "customer_previous_avg_delay": "Average payment delay from previous completed payments",
            "customer_currency_previous_avg_amount": "Historical average invoice amount for the same customer and currency",
            "customer_amount_vs_history_log": "Logarithm of current amount relative to historical same-currency average",
            "customer_no_invoice_history": "Indicates absence of previous invoice history",
            "customer_no_payment_history": "Indicates absence of previous payment history",
            "customer_no_same_currency_history": "Indicates absence of previous same-currency invoice history"
        }

        description = descriptions.get(
            original_name,
            "Numerical model input feature"
        )

    elif feature.startswith("categorical__"):
        original_name = feature.replace("categorical__", "")
        feature_group = "One-Hot Encoded Categorical"
        data_type = "Binary (0/1)"
        description = (
            "One-hot encoded category: " + original_name
        )

    elif feature == "sales_rep_frequency":
        original_name = "sales_rep"
        feature_group = "Frequency Encoded"
        data_type = "Float"
        description = (
            "Training-set frequency proportion of the sales representative"
        )

    else:
        original_name = feature
        feature_group = "Other"
        data_type = "Unknown"
        description = "Additional model input feature"

    feature_details.append({
        "Feature_Name": feature,
        "Original_Feature": original_name,
        "Feature_Group": feature_group,
        "Data_Type": data_type,
        "Description": description
    })

# Create DataFrame
detailed_feature_catalogue = pd.DataFrame(feature_details)

# Save catalogue
detailed_feature_catalogue.to_csv(
    os.path.join(processed_folder, "detailed_feature_catalogue.csv"),
    index=False
)

# Display group distribution
print("FEATURE GROUP DISTRIBUTION:")
print(
    detailed_feature_catalogue["Feature_Group"]
    .value_counts()
)

print("\nDETAILED CATALOGUE SHAPE:")
print(detailed_feature_catalogue.shape)

print("\nSAMPLE FEATURES:")
print(detailed_feature_catalogue.head(10).to_string(index=False))

print("\nDETAILED FEATURE CATALOGUE SAVED SUCCESSFULLY!")


FEATURE GROUP DISTRIBUTION:
Feature_Group
One-Hot Encoded Categorical    168
Numerical                       20
Frequency Encoded                1
Name: count, dtype: int64

DETAILED CATALOGUE SHAPE:
(189, 5)

SAMPLE FEATURES:
             Feature_Name Original_Feature Feature_Group Data_Type                                Description
        numeric__quantity         quantity     Numerical   Numeric                  Number of units purchased
      numeric__unit_price       unit_price     Numerical   Numeric                             Price per unit
numeric__discount_percent discount_percent     Numerical   Numeric                Discount percentage applied
     numeric__tax_percent      tax_percent     Numerical   Numeric                     Tax percentage applied
        numeric__subtotal         subtotal     Numerical   Numeric Invoice subtotal before additional charges
 numeric__discount_amount  discount_amount     Numerical   Numeric                      Total discount amount
   

In [62]:

# WEEK 3 - FINAL DATA LEAKAGE AUDIT

# 1. Define columns that must not be used as predictors
forbidden_predictors = [
    "payment_date",
    "payment_date_clean",
    "payment_delay_days",
    "days_to_pay",
    "paid_late_flag",
    "payment_status",
    "is_paid",
    "is_open",
    "is_overdue",
    "overdue_target",
    "paid_late",
    "delay_days"
]

# 2. Check original predictor columns
leakage_columns_found = [
    col for col in all_predictor_features
    if col in forbidden_predictors
]

print("ORIGINAL PREDICTOR LEAKAGE CHECK:")
print("Forbidden columns found:", leakage_columns_found)
print("PASS:", len(leakage_columns_found) == 0)

# 3. Check final encoded feature names
final_feature_leakage = [
    feature for feature in final_feature_names
    if any(
        feature == "numeric__" + col
        or feature == "categorical__" + col
        for col in forbidden_predictors
    )
]

print("\nFINAL ENCODED FEATURE LEAKAGE CHECK:")
print("Forbidden features found:", final_feature_leakage)
print("PASS:", len(final_feature_leakage) == 0)

# 4. Verify targets are separate from predictors
print("\nTARGET SEPARATION CHECK:")
print("Classification target:", "paid_late")
print("Regression target:", "delay_days")
print("Targets included in predictors:", 
      any(col in all_predictor_features
          for col in ["paid_late", "delay_days"]))

# 5. Verify chronological split boundaries
print("\nCHRONOLOGICAL SPLIT CHECK:")

for split_name in ["train", "validation", "test"]:
    split_dates = df.loc[
        df["data_split"] == split_name, "invoice_date"
    ]

    print(
        split_name,
        "| Start:", split_dates.min(),
        "| End:", split_dates.max()
    )

# 6. Verify target distributions
print("\nLATE PAYMENT TARGET CHECK:")
print("Train late examples:", int((y_train > 0).sum()))
print("Validation late examples:", int((y_validation > 0).sum()))
print("Test late examples:", int((y_test > 0).sum()))

# 7. Save leakage audit summary
leakage_audit = pd.DataFrame({
    "Audit_Check": [
        "Original predictor leakage",
        "Encoded feature leakage",
        "Target separation",
        "Chronological split",
        "Late payment examples"
    ],
    "Result": [
        "PASS" if len(leakage_columns_found) == 0 else "FAIL",
        "PASS" if len(final_feature_leakage) == 0 else "FAIL",
        "PASS" if not any(
            col in all_predictor_features
            for col in ["paid_late", "delay_days"]
        ) else "FAIL",
        "PASS",
        "NO POSITIVE EXAMPLES"
    ]
})

leakage_audit.to_csv(
    os.path.join(processed_folder, "leakage_audit.csv"),
    index=False
)

print("\nLEAKAGE AUDIT SAVED SUCCESSFULLY!")


ORIGINAL PREDICTOR LEAKAGE CHECK:
Forbidden columns found: []
PASS: True

FINAL ENCODED FEATURE LEAKAGE CHECK:
Forbidden features found: []
PASS: True

TARGET SEPARATION CHECK:
Classification target: paid_late
Regression target: delay_days
Targets included in predictors: False

CHRONOLOGICAL SPLIT CHECK:
train | Start: 2024-01-01 00:00:00 | End: 2024-12-31 00:00:00
validation | Start: 2025-01-01 00:00:00 | End: 2025-06-30 00:00:00
test | Start: 2025-07-01 00:00:00 | End: 2025-12-31 00:00:00

LATE PAYMENT TARGET CHECK:
Train late examples: 0
Validation late examples: 0
Test late examples: 0

LEAKAGE AUDIT SAVED SUCCESSFULLY!


In [63]:

from pathlib import Path

report_folder = Path("reports")
report_folder.mkdir(exist_ok=True)

report_content = """
# InvoiceIQ – Week 3 Feature Engineering Report

## 1. Project Overview

Project: InvoiceIQ – Invoice Payment Delay & Cash-Flow Intelligence Platform

Organisation: Orvians Technologies

Week: 3 – Feature Engineering and ML Data Preparation

Objective:
Transform the cleaned invoice dataset into machine-learning-ready
features while maintaining chronological separation and preventing
data leakage.

## 2. Dataset Information

Source dataset: Organisation-provided Global B2B Invoice and Payments Dataset.

Cleaned dataset: data/cleaned/InvoiceIQ_Cleaned_Data.csv

Total records: 2,000,000

Total original columns: 39

Invoice date range: 2024-01-01 to 2025-12-31

## 3. Feature Engineering

### Time Features

The following features were created:

- invoice_month
- invoice_weekday
- is_month_end
- payment_term_days

Payment term days represents the number of days between
invoice date and due date.

The invoice date was used as the prediction reference date.

### Customer Historical Features

The following historical features were created:

- customer_previous_invoice_count
- customer_previous_avg_amount
- customer_previous_avg_delay
- customer_previous_late_rate
- customer_no_invoice_history
- customer_no_payment_history

Historical information was restricted to information available
before the current invoice's prediction date.

Invoices generated on the same date were excluded from one
another's historical invoice calculations.

Payment history was restricted to payments completed before
the current invoice date.

### Amount Features

The following features were created:

- customer_currency_previous_avg_amount
- customer_amount_vs_history
- customer_amount_vs_history_log
- customer_no_same_currency_history

Historical amount comparisons were calculated separately
for each customer and currency.

The original total_amount distribution had positive skewness
of approximately 1.03. A logarithmic transformation of the
original amount overcorrected the skewness, so the original
amount was retained.

## 4. Target Variables

### Regression Target: delay_days

Calculated using actual cleaned payment date minus due date.

Only invoices with known completed payment outcomes were
included in regression target datasets.

Target observations:

- Early payments: 1,305,067
- On-time payments: 53,752
- Late payments: 0
- Unknown payment outcomes: 641,181

### Classification Target: paid_late

Definition:

- 1 = Payment made after due date
- 0 = Payment made on or before due date
- Missing = Payment outcome unavailable

No positive late-payment examples were present in the dataset.

Therefore, meaningful late-payment classification cannot
currently be trained or evaluated.

The dataset must not be treated as evidence of a validated
late-payment prediction model.

## 5. Chronological Data Split

The dataset was divided by invoice date without random shuffling.

| Split | Date range | Records |
|---|---|---:|
| Training | 2024-01-01 to 2024-12-31 | 1,001,116 |
| Validation | 2025-01-01 to 2025-06-30 | 495,347 |
| Testing | 2025-07-01 to 2025-12-31 | 503,537 |

Regression target availability:

| Split | Known outcomes | Unknown outcomes |
|---|---:|---:|
| Training | 680,398 | 320,718 |
| Validation | 336,301 | 159,046 |
| Testing | 342,120 | 161,417 |

## 6. Encoding and Scaling

Numerical features:
- Missing values imputed using training-set medians.
- StandardScaler fitted on training data only.

Categorical features:
- Missing values imputed using training-set most frequent values.
- OneHotEncoder used with handle_unknown='ignore'.

Frequency encoding:
- sales_rep encoded using training-set frequencies only.
- The same mapping applied to validation and testing data.
- Unseen categories handled with a frequency of zero.

The preprocessing pipeline was fitted only on training data.

## 7. Final ML-Ready Features

Total predictor features: 189

| Feature group | Count |
|---|---:|
| Numerical | 20 |
| One-hot encoded categorical | 168 |
| Frequency encoded | 1 |
| Total | 189 |

Final regression dataset shapes:

| Dataset | Feature shape |
|---|---:|
| Training | 680,398 x 189 |
| Validation | 336,301 x 189 |
| Testing | 342,120 x 189 |

Sparse matrices were used to store encoded features efficiently.

## 8. Data Leakage Audit

The following outcome-related columns were excluded
from model predictors:

- payment_date
- payment_date_clean
- payment_delay_days
- days_to_pay
- paid_late_flag
- payment_status
- is_paid
- is_open
- is_overdue
- overdue_target
- paid_late
- delay_days

Historical customer features were calculated using only
eligible prior information.

The encoded feature audit found no forbidden outcome columns.

Targets were kept separate from predictors.

## 9. Saved Deliverables

The following files were saved in data/processed/:

- X_train.npz
- X_validation.npz
- X_test.npz
- y_train.npy
- y_validation.npy
- y_test.npy
- feature_catalogue.csv
- detailed_feature_catalogue.csv
- preprocessor.joblib
- sales_rep_frequency_map.joblib
- chronological_split_summary.csv
- leakage_audit.csv

Saved feature matrices and targets were reloaded and
successfully compared with their original in-memory versions.

All six consistency checks passed.

## 10. Limitations and Assumptions

1. No late-payment examples were present in the dataset.
2. Classification modelling for paid_late is therefore not
   currently feasible.
3. Regression modelling is limited to paid invoices and
   observed early/on-time payment delays.
4. Results cannot establish performance on late payments.
5. Historical features depend on the availability and
   correctness of prior invoice and payment records.
6. The raw payment_date column was not treated as an actual
   payment outcome because it contained placeholder values.
7. The current regression dataset represents paid invoices,
   not all invoices at the time they are issued.

## 11. Conclusion

Week 3 successfully transformed the cleaned invoice dataset
into 189 machine-learning predictor features.

Chronological splitting, training-only preprocessing,
historical customer feature engineering, categorical encoding,
sparse matrix storage, and leakage checks were completed.

All saved datasets were successfully reloaded and verified.

The absence of late-payment examples was identified as a
significant dataset limitation and documented for future
modelling decisions.
"""

report_path = report_folder / "Week3_Feature_Engineering_Report.md"

report_path.write_text(
    report_content.strip(),
    encoding="utf-8"
)

print("WEEK 3 REPORT CREATED SUCCESSFULLY!")
print("FILE:", report_path.resolve())
print("FILE SIZE:", round(report_path.stat().st_size / 1024, 2), "KB")


WEEK 3 REPORT CREATED SUCCESSFULLY!
FILE: C:\Users\palar\OneDrive\Desktop\InvoiceIQ\reports\Week3_Feature_Engineering_Report.md
FILE SIZE: 6.48 KB


In [64]:

from pathlib import Path

# Expected Week 3 deliverables
expected_files = [
    "X_train.npz",
    "X_validation.npz",
    "X_test.npz",
    "y_train.npy",
    "y_validation.npy",
    "y_test.npy",
    "feature_catalogue.csv",
    "detailed_feature_catalogue.csv",
    "preprocessor.joblib",
    "sales_rep_frequency_map.joblib",
    "chronological_split_summary.csv",
    "leakage_audit.csv"
]

processed_folder = Path("data/processed")

print("WEEK 3 DELIVERABLE AUDIT\n")

all_files_present = True

for filename in expected_files:
    file_path = processed_folder / filename
    exists = file_path.exists()

    print(
        ("PASS" if exists else "MISSING"),
        "|",
        filename
    )

    if not exists:
        all_files_present = False

# Verify report
report_path = Path("reports/Week3_Feature_Engineering_Report.md")

print("\nTECHNICAL REPORT:")
print("PASS" if report_path.exists() else "MISSING")

# Final status
print("\n" + "=" * 45)

if all_files_present and report_path.exists():
    print("ALL WEEK 3 DELIVERABLES ARE PRESENT!")
else:
    print("SOME DELIVERABLES ARE MISSING!")

print("=" * 45)


WEEK 3 DELIVERABLE AUDIT

PASS | X_train.npz
PASS | X_validation.npz
PASS | X_test.npz
PASS | y_train.npy
PASS | y_validation.npy
PASS | y_test.npy
PASS | feature_catalogue.csv
PASS | detailed_feature_catalogue.csv
PASS | preprocessor.joblib
PASS | sales_rep_frequency_map.joblib
PASS | chronological_split_summary.csv
PASS | leakage_audit.csv

TECHNICAL REPORT:
PASS

ALL WEEK 3 DELIVERABLES ARE PRESENT!


In [65]:

# WEEK 3 - POINT-IN-TIME HISTORICAL FEATURE AUDIT

# Select 200 invoices for independent verification
audit_sample = df.sample(n=200, random_state=42)

# Customer row positions
customer_positions = df.groupby(
    "customer_id", sort=False
).indices

checks = {
    "previous_invoice_count": [],
    "previous_avg_amount": [],
    "previous_avg_delay": [],
    "previous_late_rate": [],
    "previous_same_currency_avg_amount": []
}

for idx, row in audit_sample.iterrows():

    customer_rows = df.iloc[
        customer_positions[row["customer_id"]]
    ]

    # Only invoices issued strictly before the current invoice
    previous_invoices = customer_rows[
        customer_rows["invoice_date"] < row["invoice_date"]
    ]

    # Previous invoice count
    expected_count = len(previous_invoices)

    checks["previous_invoice_count"].append(
        expected_count == row["customer_previous_invoice_count"]
    )

    # Previous average invoice amount
    expected_amount = (
        previous_invoices["total_amount"].mean()
        if expected_count > 0 else np.nan
    )

    checks["previous_avg_amount"].append(
        (
            pd.isna(expected_amount)
            and pd.isna(row["customer_previous_avg_amount"])
        )
        or np.isclose(
            expected_amount,
            row["customer_previous_avg_amount"],
            rtol=1e-8,
            atol=1e-8
        )
    )

    # Historical completed payments strictly before current invoice date
    previous_payments = customer_rows[
        customer_rows["payment_date_clean"].notna()
        & (
            customer_rows["payment_date_clean"]
            < row["invoice_date"]
        )
    ]

    expected_delay = previous_payments[
        "payment_delay_days"
    ].mean()

    expected_late_rate = (
        previous_payments["payment_delay_days"] > 0
    ).mean()

    checks["previous_avg_delay"].append(
        (
            pd.isna(expected_delay)
            and pd.isna(row["customer_previous_avg_delay"])
        )
        or np.isclose(
            expected_delay,
            row["customer_previous_avg_delay"],
            rtol=1e-8,
            atol=1e-8
        )
    )

    checks["previous_late_rate"].append(
        (
            pd.isna(expected_late_rate)
            and pd.isna(row["customer_previous_late_rate"])
        )
        or np.isclose(
            expected_late_rate,
            row["customer_previous_late_rate"],
            rtol=1e-8,
            atol=1e-8
        )
    )

    # Previous invoices in the same currency only
    previous_same_currency = previous_invoices[
        previous_invoices["currency"] == row["currency"]
    ]

    expected_currency_amount = (
        previous_same_currency["total_amount"].mean()
        if len(previous_same_currency) > 0 else np.nan
    )

    checks["previous_same_currency_avg_amount"].append(
        (
            pd.isna(expected_currency_amount)
            and pd.isna(
                row["customer_currency_previous_avg_amount"]
            )
        )
        or np.isclose(
            expected_currency_amount,
            row["customer_currency_previous_avg_amount"],
            rtol=1e-8,
            atol=1e-8
        )
    )

# Display results
print("POINT-IN-TIME HISTORICAL FEATURE AUDIT")
print("Sample size:", len(audit_sample))

all_checks_passed = True

for feature, results in checks.items():

    passed = sum(results)
    failed = len(results) - passed

    print("\n", feature)
    print("Passed:", passed)
    print("Failed:", failed)

    if failed > 0:
        all_checks_passed = False

print("\nFINAL RESULT:")
print(
    "ALL HISTORICAL CHECKS PASSED!"
    if all_checks_passed
    else "SOME HISTORICAL CHECKS NEED INVESTIGATION!"
)


POINT-IN-TIME HISTORICAL FEATURE AUDIT
Sample size: 200

 previous_invoice_count
Passed: 200
Failed: 0

 previous_avg_amount
Passed: 200
Failed: 0

 previous_avg_delay
Passed: 200
Failed: 0

 previous_late_rate
Passed: 200
Failed: 0

 previous_same_currency_avg_amount
Passed: 200
Failed: 0

FINAL RESULT:
ALL HISTORICAL CHECKS PASSED!


In [66]:
from pathlib import Path

# Locate the Week 3 report
report_path = Path("reports/Week3_Feature_Engineering_Report.md")

# Check whether the report exists
if report_path.exists():
    report_text = report_path.read_text(encoding="utf-8")

    print("REPORT FOUND!")
    print("Report size:", len(report_text), "characters")

    if "Point-in-Time Historical Feature Validation" in report_text:
        print("\nPoint-in-Time Audit section is already present.")
    else:
        print("\nPoint-in-Time Audit section is NOT present.")
        print("We need to add it.")

else:
    print("ERROR: Week 3 report was not found.")

REPORT FOUND!
Report size: 6387 characters

Point-in-Time Audit section is NOT present.
We need to add it.


In [67]:
from pathlib import Path

report_path = Path("reports/Week3_Feature_Engineering_Report.md")

# Read the existing report
report_text = report_path.read_text(encoding="utf-8")

# New audit section
audit_section = """

## 12. Point-in-Time Historical Feature Validation

An independent sample of 200 invoices was checked against the original dataset to validate the historical customer features.

The following five features were tested:

1. customer_previous_invoice_count
2. customer_previous_avg_amount
3. customer_previous_avg_delay
4. customer_previous_late_rate
5. customer_currency_previous_avg_amount

**Validation Results:**

| Historical Feature | Passed | Failed |
|---|---:|---:|
| Previous Invoice Count | 200 | 0 |
| Previous Average Amount | 200 | 0 |
| Previous Average Delay | 200 | 0 |
| Previous Late Rate | 200 | 0 |
| Previous Same-Currency Average Amount | 200 | 0 |
| **Total** | **1,000** | **0** |

All 1,000 comparisons passed successfully, with zero failures.

This sample-based validation provides additional evidence that the historical customer features respect the prediction-time boundary and use only information available before the current invoice date.

**Validation Limitation:** This was a sample-based audit of 200 invoices, not an exhaustive verification of all 2 million records.
"""

# Prevent duplicate sections
if "## 12. Point-in-Time Historical Feature Validation" not in report_text:
    report_path.write_text(report_text + audit_section, encoding="utf-8")
    print("SUCCESS: Point-in-Time Audit section added to the report.")
else:
    print("The audit section is already present. No changes made.")

print("Updated report size:", report_path.stat().st_size, "bytes")


SUCCESS: Point-in-Time Audit section added to the report.
Updated report size: 7770 bytes


In [68]:
from pathlib import Path

report_path = Path("reports/Week3_Feature_Engineering_Report.md")

report_text = report_path.read_text(encoding="utf-8")

# Display the newly added section
start = report_text.find("## 12. Point-in-Time Historical Feature Validation")

if start != -1:
    print("AUDIT SECTION VERIFIED!\n")
    print(report_text[start:])
else:
    print("ERROR: Audit section not found.")
    

AUDIT SECTION VERIFIED!

## 12. Point-in-Time Historical Feature Validation

An independent sample of 200 invoices was checked against the original dataset to validate the historical customer features.

The following five features were tested:

1. customer_previous_invoice_count
2. customer_previous_avg_amount
3. customer_previous_avg_delay
4. customer_previous_late_rate
5. customer_currency_previous_avg_amount

**Validation Results:**

| Historical Feature | Passed | Failed |
|---|---:|---:|
| Previous Invoice Count | 200 | 0 |
| Previous Average Amount | 200 | 0 |
| Previous Average Delay | 200 | 0 |
| Previous Late Rate | 200 | 0 |
| Previous Same-Currency Average Amount | 200 | 0 |
| **Total** | **1,000** | **0** |

All 1,000 comparisons passed successfully, with zero failures.

This sample-based validation provides additional evidence that the historical customer features respect the prediction-time boundary and use only information available before the current invoice date.

**Va

In [69]:
print("AMOUNT BUCKET IMPLEMENTATION CHECK")

amount_bucket_columns = [
    col for col in df.columns
    if "amount_bucket" in col.lower()
    or "amount_category" in col.lower()
]

if amount_bucket_columns:
    print("Amount bucket columns found:")
    print(amount_bucket_columns)
else:
    print("No amount bucket columns found.")
    print("Amount buckets have not been implemented yet.")
    

AMOUNT BUCKET IMPLEMENTATION CHECK
No amount bucket columns found.
Amount buckets have not been implemented yet.


In [70]:
required_variables = [
    "X_train_raw",
    "X_validation_raw",
    "X_test_raw"
]

print("CHECKING REQUIRED VARIABLES\n")

for variable in required_variables:
    if variable in globals():
        print(f"{variable}: AVAILABLE")
    else:
        print(f"{variable}: NOT AVAILABLE")

CHECKING REQUIRED VARIABLES

X_train_raw: AVAILABLE
X_validation_raw: AVAILABLE
X_test_raw: AVAILABLE


In [71]:
print("TRAINING DATA SHAPE:", X_train_raw.shape)
print("VALIDATION DATA SHAPE:", X_validation_raw.shape)
print("TEST DATA SHAPE:", X_test_raw.shape)

print("\nTOTAL_AMOUNT COLUMN CHECK:")

for name, dataset in [
    ("TRAIN", X_train_raw),
    ("VALIDATION", X_validation_raw),
    ("TEST", X_test_raw)
]:
    print(f"{name}: {'total_amount' in dataset.columns}")

print("\nTRAINING AMOUNT SUMMARY:")
print(X_train_raw["total_amount"].describe())


TRAINING DATA SHAPE: (680398, 30)
VALIDATION DATA SHAPE: (336301, 30)
TEST DATA SHAPE: (342120, 30)

TOTAL_AMOUNT COLUMN CHECK:
TRAIN: True
VALIDATION: True
TEST: True

TRAINING AMOUNT SUMMARY:
count    680398.000000
mean      33986.864418
std       29900.308026
min           8.710000
25%        9487.945000
50%       25372.315000
75%       51515.990000
max      159825.760000
Name: total_amount, dtype: float64


In [72]:
import numpy as np
import pandas as pd

# Calculate boundaries using training data only
amount_q1 = X_train_raw["total_amount"].quantile(0.25)
amount_q2 = X_train_raw["total_amount"].quantile(0.50)
amount_q3 = X_train_raw["total_amount"].quantile(0.75)

# Store boundaries
amount_bucket_bins = [
    -np.inf,
    amount_q1,
    amount_q2,
    amount_q3,
    np.inf
]

amount_bucket_labels = [
    "Low Amount",
    "Medium Amount",
    "High Amount",
    "Very High Amount"
]

print("AMOUNT BUCKET BOUNDARIES")
print("Q1:", amount_q1)
print("Q2:", amount_q2)
print("Q3:", amount_q3)

# Apply identical boundaries to all datasets
for dataset in [
    X_train_raw,
    X_validation_raw,
    X_test_raw
]:
    dataset["amount_bucket"] = pd.cut(
        dataset["total_amount"],
        bins=amount_bucket_bins,
        labels=amount_bucket_labels,
        include_lowest=True
    )

print("\nAMOUNT BUCKETS CREATED SUCCESSFULLY!")

AMOUNT BUCKET BOUNDARIES
Q1: 9487.945
Q2: 25372.315000000002
Q3: 51515.99

AMOUNT BUCKETS CREATED SUCCESSFULLY!


In [73]:
datasets = {
    "TRAIN": X_train_raw,
    "VALIDATION": X_validation_raw,
    "TEST": X_test_raw
}

for name, dataset in datasets.items():
    print(f"\n{'='*15} {name} {'='*15}")

    print("\nAmount Bucket Distribution:")
    print(dataset["amount_bucket"].value_counts().sort_index())

    print("\nMissing Bucket Values:")
    print(dataset["amount_bucket"].isna().sum())


=============== TRAIN ===============

Amount Bucket Distribution:
amount_bucket
Low Amount          170100
Medium Amount       170099
High Amount         170099
Very High Amount    170100
Name: count, dtype: int64

Missing Bucket Values:
0

=============== VALIDATION ===============

Amount Bucket Distribution:
amount_bucket
Low Amount          83974
Medium Amount       84357
High Amount         84228
Very High Amount    83742
Name: count, dtype: int64

Missing Bucket Values:
0

=============== TEST ===============

Amount Bucket Distribution:
amount_bucket
Low Amount          85675
Medium Amount       85282
High Amount         85653
Very High Amount    85510
Name: count, dtype: int64

Missing Bucket Values:
0


In [74]:
print("CURRENT ONE-HOT FEATURES:")
print(onehot_features)

print("\nCURRENT NUMERICAL FEATURES:")
print(numeric_features)

print("\nPREPROCESSOR AVAILABLE:")
print("preprocessor" in globals())

print("\nENCODED FEATURE COUNT:")
print(len(all_predictor_features))

print("\nAMOUNT BUCKET DATA TYPE:")
print(X_train_raw["amount_bucket"].dtype)

CURRENT ONE-HOT FEATURES:
['currency', 'payment_method', 'order_channel', 'product_category', 'product_name', 'vendor_id', 'customer_country', 'customer_state', 'customer_city']

CURRENT NUMERICAL FEATURES:
['quantity', 'unit_price', 'discount_percent', 'tax_percent', 'subtotal', 'discount_amount', 'tax_amount', 'shipping_cost', 'total_amount', 'invoice_month', 'invoice_weekday', 'is_month_end', 'payment_term_days', 'customer_previous_invoice_count', 'customer_previous_avg_delay', 'customer_currency_previous_avg_amount', 'customer_amount_vs_history_log', 'customer_no_invoice_history', 'customer_no_payment_history', 'customer_no_same_currency_history']

PREPROCESSOR AVAILABLE:
True

ENCODED FEATURE COUNT:
30

AMOUNT BUCKET DATA TYPE:
category


In [75]:
print("PREPROCESSOR CONFIGURATION:\n")

for name, transformer, columns in preprocessor.transformers:
    print("Transformer:", name)
    print("Columns:", columns)
    print("Transformer details:", transformer)
    print("-" * 60)
    

PREPROCESSOR CONFIGURATION:

Transformer: numeric
Columns: ['quantity', 'unit_price', 'discount_percent', 'tax_percent', 'subtotal', 'discount_amount', 'tax_amount', 'shipping_cost', 'total_amount', 'invoice_month', 'invoice_weekday', 'is_month_end', 'payment_term_days', 'customer_previous_invoice_count', 'customer_previous_avg_delay', 'customer_currency_previous_avg_amount', 'customer_amount_vs_history_log', 'customer_no_invoice_history', 'customer_no_payment_history', 'customer_no_same_currency_history']
Transformer details: Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])
------------------------------------------------------------
Transformer: categorical
Columns: ['currency', 'payment_method', 'order_channel', 'product_category', 'product_name', 'vendor_id', 'customer_country', 'customer_state', 'customer_city']
Transformer details: Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('o

In [76]:
# Add amount_bucket to categorical features
if "amount_bucket" not in onehot_features:
    onehot_features.append("amount_bucket")

# Rebuild the complete predictor feature list
all_predictor_features = (
    numeric_features
    + onehot_features
    + frequency_features
)

print("UPDATED FEATURE CONFIGURATION")

print("\nNumerical Features:", len(numeric_features))
print("Categorical Features:", len(onehot_features))
print("Frequency Features:", len(frequency_features))

print("\nTOTAL RAW PREDICTOR FEATURES:",
      len(all_predictor_features))

print("\nUPDATED CATEGORICAL FEATURES:")
print(onehot_features)

# Check for duplicate feature names
print("\nDUPLICATE FEATURE NAMES:",
      len(all_predictor_features) - len(set(all_predictor_features)))

UPDATED FEATURE CONFIGURATION

Numerical Features: 20
Categorical Features: 10
Frequency Features: 1

TOTAL RAW PREDICTOR FEATURES: 31

UPDATED CATEGORICAL FEATURES:
['currency', 'payment_method', 'order_channel', 'product_category', 'product_name', 'vendor_id', 'customer_country', 'customer_state', 'customer_city', 'amount_bucket']

DUPLICATE FEATURE NAMES: 0


In [77]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical preprocessing
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

# Combine both pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, onehot_features)
    ],
    remainder="drop"
)

print("UPDATED PREPROCESSOR CREATED SUCCESSFULLY!")

print("\nNumerical columns:", len(numeric_features))
print("Categorical columns:", len(onehot_features))
print("Amount bucket included:",
      "amount_bucket" in onehot_features)

UPDATED PREPROCESSOR CREATED SUCCESSFULLY!

Numerical columns: 20
Categorical columns: 10
Amount bucket included: True


In [78]:
# Fit preprocessor using training data only
X_train_processed_base = preprocessor.fit_transform(
    X_train_raw[numeric_features + onehot_features]
)

# Transform validation and test using the fitted preprocessor
X_validation_processed_base = preprocessor.transform(
    X_validation_raw[numeric_features + onehot_features]
)

X_test_processed_base = preprocessor.transform(
    X_test_raw[numeric_features + onehot_features]
)

print("PREPROCESSING COMPLETED!")

print("\nTRAIN MATRIX SHAPE:",
      X_train_processed_base.shape)

print("VALIDATION MATRIX SHAPE:",
      X_validation_processed_base.shape)

print("TEST MATRIX SHAPE:",
      X_test_processed_base.shape)

PREPROCESSING COMPLETED!

TRAIN MATRIX SHAPE: (680398, 192)
VALIDATION MATRIX SHAPE: (336301, 192)
TEST MATRIX SHAPE: (342120, 192)


In [79]:
print("FREQUENCY ENCODING VERIFICATION")

print("\nTRAIN FREQUENCY SHAPE:",
      X_train_frequency.shape)

print("VALIDATION FREQUENCY SHAPE:",
      X_validation_frequency.shape)

print("TEST FREQUENCY SHAPE:",
      X_test_frequency.shape)

print("\nFREQUENCY MAP SIZE:",
      len(sales_rep_frequency_map))

print("\nMISSING VALUES:")
print("Train:", X_train_frequency.isna().sum())
print("Validation:", X_validation_frequency.isna().sum())
print("Test:", X_test_frequency.isna().sum())

FREQUENCY ENCODING VERIFICATION

TRAIN FREQUENCY SHAPE: (680398,)
VALIDATION FREQUENCY SHAPE: (336301,)
TEST FREQUENCY SHAPE: (342120,)

FREQUENCY MAP SIZE: 233

MISSING VALUES:
Train: 0
Validation: 0
Test: 0


In [80]:
from scipy.sparse import hstack, csr_matrix

# Convert frequency arrays into sparse column matrices
X_train_frequency_sparse = csr_matrix(
    X_train_frequency.to_numpy().reshape(-1, 1)
)

X_validation_frequency_sparse = csr_matrix(
    X_validation_frequency.to_numpy().reshape(-1, 1)
)

X_test_frequency_sparse = csr_matrix(
    X_test_frequency.to_numpy().reshape(-1, 1)
)

# Combine preprocessing output and frequency encoding
X_train_processed = hstack(
    [X_train_processed_base, X_train_frequency_sparse],
    format="csr"
)

X_validation_processed = hstack(
    [X_validation_processed_base, X_validation_frequency_sparse],
    format="csr"
)

X_test_processed = hstack(
    [X_test_processed_base, X_test_frequency_sparse],
    format="csr"
)

print("FINAL ML-READY MATRICES CREATED!")

print("\nTRAIN:", X_train_processed.shape)
print("VALIDATION:", X_validation_processed.shape)
print("TEST:", X_test_processed.shape)

print("\nSPARSE MATRIX CHECK:")
print("Train:", X_train_processed.format)
print("Validation:", X_validation_processed.format)
print("Test:", X_test_processed.format)

FINAL ML-READY MATRICES CREATED!

TRAIN: (680398, 193)
VALIDATION: (336301, 193)
TEST: (342120, 193)

SPARSE MATRIX CHECK:
Train: csr
Validation: csr
Test: csr


In [81]:
# Get feature names from the updated preprocessor
updated_encoded_feature_names = (
    preprocessor.get_feature_names_out().tolist()
)

# Add frequency feature name
updated_final_feature_names = (
    updated_encoded_feature_names
    + ["sales_rep_frequency"]
)

print("UPDATED FEATURE CATALOGUE")

print("\nTOTAL FEATURES:",
      len(updated_final_feature_names))

print("\nUNIQUE FEATURES:",
      len(set(updated_final_feature_names)))

print("\nAMOUNT BUCKET ENCODED FEATURES:")

for feature in updated_final_feature_names:
    if "amount_bucket" in feature:
        print(feature)

print("\nLAST 10 FEATURES:")
print(updated_final_feature_names[-10:])

print("\nDUPLICATE FEATURE NAMES:",
      len(updated_final_feature_names)
      - len(set(updated_final_feature_names)))

print("\nMATRIX FEATURE COUNT:",
      X_train_processed.shape[1])

UPDATED FEATURE CATALOGUE

TOTAL FEATURES: 193

UNIQUE FEATURES: 193

AMOUNT BUCKET ENCODED FEATURES:
categorical__amount_bucket_High Amount
categorical__amount_bucket_Low Amount
categorical__amount_bucket_Medium Amount
categorical__amount_bucket_Very High Amount

LAST 10 FEATURES:
['categorical__customer_city_Singapore', 'categorical__customer_city_Sydney', 'categorical__customer_city_Tokyo', 'categorical__customer_city_Toronto', 'categorical__customer_city_Vancouver', 'categorical__amount_bucket_High Amount', 'categorical__amount_bucket_Low Amount', 'categorical__amount_bucket_Medium Amount', 'categorical__amount_bucket_Very High Amount', 'sales_rep_frequency']

DUPLICATE FEATURE NAMES: 0

MATRIX FEATURE COUNT: 193


In [82]:
from pathlib import Path
import pandas as pd

# Create feature catalogue DataFrame
updated_feature_catalogue = pd.DataFrame({
    "Feature_Number": range(1, len(updated_final_feature_names) + 1),
    "Feature_Name": updated_final_feature_names
})

# Create output folder
processed_folder = Path("data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

# Save updated feature catalogue
updated_feature_catalogue.to_csv(
    processed_folder / "feature_catalogue.csv",
    index=False
)

print("UPDATED FEATURE CATALOGUE SAVED!")

print("\nTOTAL FEATURES:",
      len(updated_feature_catalogue))

print("\nSAVED FILE:")
print(processed_folder / "feature_catalogue.csv")

print("\nFIRST 5 FEATURES:")
print(updated_feature_catalogue.head())

print("\nLAST 5 FEATURES:")
print(updated_feature_catalogue.tail())

UPDATED FEATURE CATALOGUE SAVED!

TOTAL FEATURES: 193

SAVED FILE:
data\processed\feature_catalogue.csv

FIRST 5 FEATURES:
   Feature_Number               Feature_Name
0               1          numeric__quantity
1               2        numeric__unit_price
2               3  numeric__discount_percent
3               4       numeric__tax_percent
4               5          numeric__subtotal

LAST 5 FEATURES:
     Feature_Number                                 Feature_Name
188             189       categorical__amount_bucket_High Amount
189             190        categorical__amount_bucket_Low Amount
190             191     categorical__amount_bucket_Medium Amount
191             192  categorical__amount_bucket_Very High Amount
192             193                          sales_rep_frequency


In [83]:
# Create detailed feature catalogue
feature_groups = []

for feature in updated_final_feature_names:

    if feature.startswith("numeric__"):
        group = "Numerical"
        source = feature.replace("numeric__", "")
        preprocessing = "Median Imputation + StandardScaler"

    elif feature.startswith("categorical__"):
        group = "Categorical"
        source = feature.replace("categorical__", "")
        preprocessing = "Most Frequent Imputation + One-Hot Encoding"

    elif feature == "sales_rep_frequency":
        group = "Frequency Encoded"
        source = "sales_rep"
        preprocessing = "Training Data Frequency Encoding"

    else:
        group = "Unknown"
        source = feature
        preprocessing = "Not specified"

    feature_groups.append({
        "Feature_Number": len(feature_groups) + 1,
        "Feature_Name": feature,
        "Feature_Group": group,
        "Source_Column": source,
        "Preprocessing_Method": preprocessing
    })

updated_detailed_catalogue = pd.DataFrame(feature_groups)

# Save detailed catalogue
updated_detailed_catalogue.to_csv(
    processed_folder / "detailed_feature_catalogue.csv",
    index=False
)

print("DETAILED FEATURE CATALOGUE SAVED!")

print("\nSHAPE:", updated_detailed_catalogue.shape)

print("\nFEATURE GROUP DISTRIBUTION:")
print(updated_detailed_catalogue["Feature_Group"].value_counts())

print("\nAMOUNT BUCKET FEATURES:")
print(
    updated_detailed_catalogue[
        updated_detailed_catalogue["Source_Column"].str.contains(
            "amount_bucket", case=False
        )
    ]
)

DETAILED FEATURE CATALOGUE SAVED!

SHAPE: (193, 5)

FEATURE GROUP DISTRIBUTION:
Feature_Group
Categorical          172
Numerical             20
Frequency Encoded      1
Name: count, dtype: int64

AMOUNT BUCKET FEATURES:
     Feature_Number                                 Feature_Name  \
188             189       categorical__amount_bucket_High Amount   
189             190        categorical__amount_bucket_Low Amount   
190             191     categorical__amount_bucket_Medium Amount   
191             192  categorical__amount_bucket_Very High Amount   

    Feature_Group                   Source_Column  \
188   Categorical       amount_bucket_High Amount   
189   Categorical        amount_bucket_Low Amount   
190   Categorical     amount_bucket_Medium Amount   
191   Categorical  amount_bucket_Very High Amount   

                            Preprocessing_Method  
188  Most Frequent Imputation + One-Hot Encoding  
189  Most Frequent Imputation + One-Hot Encoding  
190  Most Frequent I

In [84]:
import numpy as np
import joblib
from scipy.sparse import save_npz
from pathlib import Path

# Output folder
processed_folder = Path("data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

# 1. Save updated feature matrices
save_npz(processed_folder / "X_train.npz", X_train_processed)
save_npz(processed_folder / "X_validation.npz", X_validation_processed)
save_npz(processed_folder / "X_test.npz", X_test_processed)

# 2. Save regression targets
np.save(processed_folder / "y_train.npy", y_train)
np.save(processed_folder / "y_validation.npy", y_validation)
np.save(processed_folder / "y_test.npy", y_test)

# 3. Save updated preprocessing pipeline
joblib.dump(
    preprocessor,
    processed_folder / "preprocessor.joblib"
)

# 4. Save frequency encoding map
joblib.dump(
    sales_rep_frequency_map,
    processed_folder / "sales_rep_frequency_map.joblib"
)

# 5. Save amount bucket configuration
amount_bucket_config = {
    "bins": amount_bucket_bins,
    "labels": amount_bucket_labels,
    "boundary_source": "Training data only",
    "method": "Training data quartiles"
}

joblib.dump(
    amount_bucket_config,
    processed_folder / "amount_bucket_config.joblib"
)

print("ALL UPDATED ML-READY FILES SAVED SUCCESSFULLY!")

print("\nTRAIN MATRIX:", X_train_processed.shape)
print("VALIDATION MATRIX:", X_validation_processed.shape)
print("TEST MATRIX:", X_test_processed.shape)

print("\nTARGET SHAPES:")
print("Train:", y_train.shape)
print("Validation:", y_validation.shape)
print("Test:", y_test.shape)

print("\nUPDATED PREPROCESSOR SAVED")
print("FREQUENCY MAP SAVED")
print("AMOUNT BUCKET CONFIGURATION SAVED")

ALL UPDATED ML-READY FILES SAVED SUCCESSFULLY!

TRAIN MATRIX: (680398, 193)
VALIDATION MATRIX: (336301, 193)
TEST MATRIX: (342120, 193)

TARGET SHAPES:
Train: (680398,)
Validation: (336301,)
Test: (342120,)

UPDATED PREPROCESSOR SAVED
FREQUENCY MAP SAVED
AMOUNT BUCKET CONFIGURATION SAVED


In [85]:
import numpy as np
import joblib
from scipy.sparse import load_npz
from pathlib import Path

processed_folder = Path("data/processed")

# Reload feature matrices
loaded_X_train = load_npz(processed_folder / "X_train.npz")
loaded_X_validation = load_npz(processed_folder / "X_validation.npz")
loaded_X_test = load_npz(processed_folder / "X_test.npz")

# Reload targets
loaded_y_train = np.load(processed_folder / "y_train.npy")
loaded_y_validation = np.load(processed_folder / "y_validation.npy")
loaded_y_test = np.load(processed_folder / "y_test.npy")

# Reload preprocessing objects
loaded_preprocessor = joblib.load(
    processed_folder / "preprocessor.joblib"
)

loaded_frequency_map = joblib.load(
    processed_folder / "sales_rep_frequency_map.joblib"
)

loaded_amount_bucket_config = joblib.load(
    processed_folder / "amount_bucket_config.joblib"
)

print("RELOAD VERIFICATION RESULTS\n")

checks = {
    "Train matrix": loaded_X_train.shape == (680398, 193),
    "Validation matrix": loaded_X_validation.shape == (336301, 193),
    "Test matrix": loaded_X_test.shape == (342120, 193),
    "Train target": loaded_y_train.shape == (680398,),
    "Validation target": loaded_y_validation.shape == (336301,),
    "Test target": loaded_y_test.shape == (342120,),
    "Preprocessor": loaded_preprocessor is not None,
    "Frequency map": len(loaded_frequency_map) == 233,
    "Amount bucket configuration": len(loaded_amount_bucket_config["labels"]) == 4
}

for name, result in checks.items():
    print(f"{name}: {'PASS' if result else 'FAIL'}")

print("\nTOTAL CHECKS:", len(checks))
print("PASSED:", sum(checks.values()))
print("FAILED:", len(checks) - sum(checks.values()))

RELOAD VERIFICATION RESULTS

Train matrix: PASS
Validation matrix: PASS
Test matrix: PASS
Train target: PASS
Validation target: PASS
Test target: PASS
Preprocessor: PASS
Frequency map: PASS
Amount bucket configuration: PASS

TOTAL CHECKS: 9
PASSED: 9
FAILED: 0


In [86]:
from pathlib import Path

report_path = Path("reports/Week3_Feature_Engineering_Report.md")

report_text = report_path.read_text(encoding="utf-8")

print("CHECKING OLD FEATURE REFERENCES\n")

for line_number, line in enumerate(report_text.splitlines(), start=1):
    if any(term in line.lower() for term in [
        "189",
        "30 raw",
        "20 numerical",
        "168 one-hot",
        "amount bucket",
        "saved deliverables"
    ]):
        print(f"Line {line_number}: {line}")

print("\nREPORT CHECK COMPLETED.")

CHECKING OLD FEATURE REFERENCES

Line 150: Total predictor features: 189
Line 157: | Total | 189 |
Line 163: | Training | 680,398 x 189 |
Line 164: | Validation | 336,301 x 189 |
Line 165: | Testing | 342,120 x 189 |
Line 194: ## 9. Saved Deliverables
Line 234: into 189 machine-learning predictor features.

REPORT CHECK COMPLETED.


In [87]:
from pathlib import Path

report_path = Path("reports/Week3_Feature_Engineering_Report.md")
report_lines = report_path.read_text(encoding="utf-8").splitlines()

print("REPORT SECTIONS TO REVIEW\n")

for line_number in range(135, min(245, len(report_lines)) + 1):
    print(f"{line_number}: {report_lines[line_number - 1]}")

print("\nREPORT REVIEW COMPLETED.")

REPORT SECTIONS TO REVIEW

135: - StandardScaler fitted on training data only.
136: 
137: Categorical features:
138: - Missing values imputed using training-set most frequent values.
139: - OneHotEncoder used with handle_unknown='ignore'.
140: 
141: Frequency encoding:
142: - sales_rep encoded using training-set frequencies only.
143: - The same mapping applied to validation and testing data.
144: - Unseen categories handled with a frequency of zero.
145: 
146: The preprocessing pipeline was fitted only on training data.
147: 
148: ## 7. Final ML-Ready Features
149: 
150: Total predictor features: 189
151: 
152: | Feature group | Count |
153: |---|---:|
154: | Numerical | 20 |
155: | One-hot encoded categorical | 168 |
156: | Frequency encoded | 1 |
157: | Total | 189 |
158: 
159: Final regression dataset shapes:
160: 
161: | Dataset | Feature shape |
162: |---|---:|
163: | Training | 680,398 x 189 |
164: | Validation | 336,301 x 189 |
165: | Testing | 342,120 x 189 |
166: 
167: Sparse

In [88]:
from pathlib import Path

report_path = Path("reports/Week3_Feature_Engineering_Report.md")

report_text = report_path.read_text(encoding="utf-8")

# --------------------------------------------------
# UPDATE SECTION 7: FINAL ML-READY FEATURES
# --------------------------------------------------

old_section_7_start = "## 7. Final ML-Ready Features"
old_section_8_start = "## 8. Data Leakage Audit"

new_section_7 = """## 7. Final ML-Ready Features

The final pipeline contains 31 raw predictor columns before
encoding:

| Raw feature group | Count |
|---|---:|
| Numerical | 20 |
| Categorical | 10 |
| Frequency encoded | 1 |
| Total raw predictors | 31 |

After preprocessing and encoding, the final feature matrices
contain 193 machine-learning predictor features.

| Encoded feature group | Count |
|---|---:|
| Numerical | 20 |
| One-hot encoded categorical | 172 |
| Frequency encoded | 1 |
| Total | 193 |

The categorical feature count includes four amount-bucket
categories.

### Amount Bucket Engineering

The total_amount feature was divided into four categories
using quartile boundaries calculated exclusively from the
training data.

| Boundary | Training-set value |
|---|---:|
| Q1 | 9,487.945 |
| Q2 (Median) | 25,372.315 |
| Q3 | 51,515.99 |

The resulting categories are:

- Low Amount
- Medium Amount
- High Amount
- Very High Amount

The same training-derived boundaries were applied to
training, validation, and testing datasets. No validation
or testing information was used to calculate the boundaries.

The bucket configuration was saved separately as
amount_bucket_config.joblib.

Final regression dataset shapes:

| Dataset | Feature shape |
|---|---:|
| Training | 680,398 x 193 |
| Validation | 336,301 x 193 |
| Testing | 342,120 x 193 |

All feature matrices were stored as sparse CSR matrices
for efficient memory usage.

"""

assert old_section_7_start in report_text
assert old_section_8_start in report_text

before_section_7, remaining = report_text.split(
    old_section_7_start, 1
)

_, after_section_7 = remaining.split(
    old_section_8_start, 1
)

report_text = (
    before_section_7
    + new_section_7
    + old_section_8_start
    + after_section_7
)

# --------------------------------------------------
# UPDATE SECTION 9: SAVED DELIVERABLES
# --------------------------------------------------

old_section_9_start = "## 9. Saved Deliverables"
old_section_10_start = "## 10. Limitations and Assumptions"

new_section_9 = """## 9. Saved Deliverables

The following 13 files were saved in data/processed/:

- X_train.npz
- X_validation.npz
- X_test.npz
- y_train.npy
- y_validation.npy
- y_test.npy
- feature_catalogue.csv
- detailed_feature_catalogue.csv
- preprocessor.joblib
- sales_rep_frequency_map.joblib
- chronological_split_summary.csv
- leakage_audit.csv
- amount_bucket_config.joblib

The saved feature matrices, targets, preprocessing pipeline,
frequency mapping, and amount-bucket configuration were
reloaded and verified.

All nine reload verification checks passed:

1. Training feature matrix
2. Validation feature matrix
3. Testing feature matrix
4. Training target
5. Validation target
6. Testing target
7. Preprocessor
8. Sales representative frequency mapping
9. Amount-bucket configuration

Verification result: 9 passed, 0 failed.

"""

assert old_section_9_start in report_text
assert old_section_10_start in report_text

before_section_9, remaining = report_text.split(
    old_section_9_start, 1
)

_, after_section_9 = remaining.split(
    old_section_10_start, 1
)

report_text = (
    before_section_9
    + new_section_9
    + old_section_10_start
    + after_section_9
)

# --------------------------------------------------
# UPDATE SECTION 11: CONCLUSION
# --------------------------------------------------

report_text = report_text.replace(
    "into 189 machine-learning predictor features.",
    "into 193 machine-learning predictor features."
)

# --------------------------------------------------
# SAVE UPDATED REPORT
# --------------------------------------------------

report_path.write_text(report_text, encoding="utf-8")

print("WEEK 3 REPORT UPDATED SUCCESSFULLY.")
print("Final predictor features: 193")
print("Raw predictor columns: 31")
print("Saved deliverables: 13")
print("Reload verification: 9 passed, 0 failed")

WEEK 3 REPORT UPDATED SUCCESSFULLY.
Final predictor features: 193
Raw predictor columns: 31
Saved deliverables: 13
Reload verification: 9 passed, 0 failed


In [89]:
from pathlib import Path

report_path = Path("reports/Week3_Feature_Engineering_Report.md")
report_text = report_path.read_text(encoding="utf-8")

print("FINAL REPORT VERIFICATION\n")

checks = {
    "193 final predictor features": "193 machine-learning predictor features" in report_text,
    "20 numerical features": "| Numerical | 20 |" in report_text,
    "172 categorical features": "| One-hot encoded categorical | 172 |" in report_text,
    "1 frequency feature": "| Frequency encoded | 1 |" in report_text,
    "Training shape updated": "680,398 x 193" in report_text,
    "Validation shape updated": "336,301 x 193" in report_text,
    "Testing shape updated": "342,120 x 193" in report_text,
    "Amount bucket configuration included": "amount_bucket_config.joblib" in report_text,
    "Nine verification checks documented": "9 passed, 0 failed" in report_text,
}

for check_name, result in checks.items():
    print(f"{check_name}: {'PASS' if result else 'FAIL'}")

print("\nTOTAL CHECKS:", len(checks))
print("PASSED:", sum(checks.values()))
print("FAILED:", len(checks) - sum(checks.values()))

FINAL REPORT VERIFICATION

193 final predictor features: PASS
20 numerical features: PASS
172 categorical features: PASS
1 frequency feature: PASS
Training shape updated: PASS
Validation shape updated: PASS
Testing shape updated: PASS
Amount bucket configuration included: PASS
Nine verification checks documented: PASS

TOTAL CHECKS: 9
PASSED: 9
FAILED: 0


In [90]:
from pathlib import Path

readme_path = Path("README.md")

if readme_path.exists():
    readme_text = readme_path.read_text(encoding="utf-8")

    print("CURRENT README CONTENT\n")
    print(readme_text)

    print("\nREADME LENGTH:", len(readme_text), "characters")
else:
    print("README.md NOT FOUND.")

CURRENT README CONTENT

# InvoiceIQ – Invoice Payment Delay & Cash-Flow Intelligence Platform

## Project Overview

InvoiceIQ is an AI/ML project focused on analysing invoice and payment information to understand payment behaviour, identify overdue invoices, and support cash-flow intelligence.

This project is being developed as part of an 8-week AI/ML internship.

## Week 1 – Data Collection and Cleaning

The objective of Week 1 was to transform raw invoice data into a clean, structured and analysis-ready dataset.

### Dataset Information

- Total records: 2,000,000
- Original columns: 31
- Cleaned columns: 39
- Missing values in original columns: 0
- Duplicate invoice IDs: 0
- Duplicate complete rows: 0

### Technologies Used

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Jupyter Notebook
- VS Code

### Data Cleaning Activities

1. Loaded the original invoice dataset using Pandas.
2. Inspected dataset dimensions and data types.
3. Checked missing values and duplicate records.
4.

In [91]:
from pathlib import Path

readme_path = Path("README.md")
readme_text = readme_path.read_text(encoding="utf-8")

# Find the first Next Steps section
next_steps_marker = "## Next Steps"

assert next_steps_marker in readme_text, "Next Steps section not found."

# Preserve everything before the first Next Steps section
main_content = readme_text.split(next_steps_marker, 1)[0].rstrip()

# New Week 3 documentation and project ending
new_ending = """

## Week 3 – Feature Engineering and ML Dataset Preparation

The objective of Week 3 was to transform the cleaned invoice dataset into machine-learning-ready features while maintaining chronological separation, preventing data leakage, and preparing datasets for regression modelling.

### Feature Engineering Activities

1. Created invoice time-based features, including invoice month, weekday, month-end indicator and payment-term days.
2. Generated historical customer behaviour features using only eligible previous invoices and payment records.
3. Created customer and same-currency historical amount features.
4. Developed amount buckets using quartile boundaries calculated exclusively from the training dataset.
5. Applied chronological train, validation and test splitting based on invoice dates.
6. Built numerical, categorical and frequency-encoding preprocessing pipelines.
7. Applied training-only imputation, scaling, categorical encoding and frequency mapping.
8. Audited predictor features for target leakage.
9. Saved sparse feature matrices, target arrays, preprocessing objects and feature catalogues.
10. Reloaded saved deliverables and verified their consistency.

### Chronological Data Split

| Dataset | Records | Invoice Date Range |
|---|---:|---|
| Training | 1,001,116 | 2024-01-01 to 2024-12-31 |
| Validation | 495,347 | 2025-01-01 to 2025-06-30 |
| Testing | 503,537 | 2025-07-01 to 2025-12-31 |

The regression datasets contain paid invoices with known payment-delay targets.

| Regression dataset | Records |
|---|---:|
| Training | 680,398 |
| Validation | 336,301 |
| Testing | 342,120 |

### Final ML Feature Set

The final pipeline contains 31 raw predictor columns before encoding.

| Raw feature group | Count |
|---|---:|
| Numerical | 20 |
| Categorical | 10 |
| Frequency encoded | 1 |
| Total | 31 |

After preprocessing, the final feature matrices contain 193 predictor features.

| Encoded feature group | Count |
|---|---:|
| Numerical | 20 |
| One-hot encoded categorical | 172 |
| Frequency encoded | 1 |
| Total | 193 |

The final feature matrices are stored in sparse CSR format.

### Amount Bucket Engineering

The total invoice amount was divided into four categories:

- Low Amount
- Medium Amount
- High Amount
- Very High Amount

The quartile boundaries were calculated using training data only:

- Q1: 9,487.945
- Q2: 25,372.315
- Q3: 51,515.99

The same boundaries were applied to the training, validation and testing datasets.

### Leakage Prevention and Validation

- Chronological splitting was used to preserve the time order of invoices.
- Preprocessing components were fitted using training data only.
- Historical customer features used eligible earlier records.
- Same-day records were excluded from historical calculations.
- Outcome-related columns were excluded from model predictors.
- An independent point-in-time audit checked 1,000 feature comparisons, with zero failures.
- Saved deliverables passed all nine reload verification checks.

### Week 3 Deliverables

The following 13 files were saved in `data/processed/`:

- X_train.npz
- X_validation.npz
- X_test.npz
- y_train.npy
- y_validation.npy
- y_test.npy
- feature_catalogue.csv
- detailed_feature_catalogue.csv
- preprocessor.joblib
- sales_rep_frequency_map.joblib
- chronological_split_summary.csv
- leakage_audit.csv
- amount_bucket_config.joblib

Additional documentation:

- Week 3 Feature Engineering Jupyter Notebook.
- Week 3 Feature Engineering Report.
- Updated project README.

### Important Modelling Limitation

The verified paid invoices contain no positive late-payment examples. The observed payment-delay target ranges from -60 to 0 days.

Therefore:

- A reliable paid-late classification model cannot currently be trained.
- Regression modelling is limited to observed early and on-time payment delays.
- Model performance on actual late payments cannot be established using the current dataset.
- Additional representative late-payment data may be required.

## Upcoming Work

Future tasks will follow the internship organisation's assigned Week 4 requirements and subsequent project milestones.

---

**Project:** InvoiceIQ  
**Internship:** AI/ML Internship  
**Current Stage:** Week 3 – Feature Engineering and ML Dataset Preparation
"""

# Save updated README
updated_readme = main_content + "\n" + new_ending

readme_path.write_text(updated_readme, encoding="utf-8")

print("README UPDATED SUCCESSFULLY.")
print("Week 1 content preserved.")
print("Week 2 content preserved.")
print("Week 3 documentation added.")
print("Duplicate Next Steps sections removed.")
print("Project stage updated to Week 3.")

README UPDATED SUCCESSFULLY.
Week 1 content preserved.
Week 2 content preserved.
Week 3 documentation added.
Duplicate Next Steps sections removed.
Project stage updated to Week 3.


In [92]:
from pathlib import Path

readme_text = Path("README.md").read_text(encoding="utf-8")

checks = {
    "Week 1 documentation preserved": "## Week 1" in readme_text,
    "Week 2 documentation preserved": "## Week 2" in readme_text,
    "Week 3 documentation added": "## Week 3" in readme_text,
    "193 features documented": "193 predictor features" in readme_text,
    "172 categorical features documented": "| One-hot encoded categorical | 172 |" in readme_text,
    "Amount buckets documented": "Very High Amount" in readme_text,
    "Chronological split documented": "1,001,116" in readme_text,
    "13 deliverables documented": "amount_bucket_config.joblib" in readme_text,
    "Point-in-time audit documented": "1,000 feature comparisons" in readme_text,
    "Current stage updated": "**Current Stage:** Week 3" in readme_text,
    "Only one Next Steps section remains": readme_text.count("## Next Steps") == 0,
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nTOTAL:", len(checks))
print("PASSED:", sum(checks.values()))
print("FAILED:", len(checks) - sum(checks.values()))

Week 1 documentation preserved: PASS
Week 2 documentation preserved: PASS
Week 3 documentation added: PASS
193 features documented: PASS
172 categorical features documented: PASS
Amount buckets documented: PASS
Chronological split documented: PASS
13 deliverables documented: PASS
Point-in-time audit documented: PASS
Current stage updated: PASS
Only one Next Steps section remains: PASS

TOTAL: 11
PASSED: 11
FAILED: 0


In [93]:
from pathlib import Path
import subprocess

project_path = Path.cwd()

print("PROJECT LOCATION:")
print(project_path)

# 1. Check important Week 3 files
important_files = [
    "README.md",
    "notebooks/InvoiceIQ_Week3_Feature_Engineering.ipynb",
    "reports/Week3_Feature_Engineering_Report.md",
    "data/processed/X_train.npz",
    "data/processed/X_validation.npz",
    "data/processed/X_test.npz",
    "data/processed/amount_bucket_config.joblib",
]

print("\nIMPORTANT FILE CHECK")
for file in important_files:
    path = project_path / file
    print(
        f"{file}: "
        f"{'EXISTS' if path.exists() else 'MISSING'}"
    )

# 2. Check processed file sizes
processed_path = project_path / "data" / "processed"

print("\nPROCESSED FILES")
if processed_path.exists():
    for file in sorted(processed_path.iterdir()):
        if file.is_file():
            size_mb = file.stat().st_size / (1024 * 1024)
            print(f"{file.name}: {size_mb:.2f} MB")

# 3. Display .gitignore
gitignore_path = project_path / ".gitignore"

print("\n.GITIGNORE CONTENT")
if gitignore_path.exists():
    print(gitignore_path.read_text(encoding="utf-8"))
else:
    print(".gitignore NOT FOUND")

# 4. Check Git status
print("\nGIT STATUS")
result = subprocess.run(
    ["git", "status", "--short"],
    cwd=project_path,
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else "No uncommitted changes.")
if result.stderr:
    print("Git error:", result.stderr)

print("\nPROJECT INSPECTION COMPLETED.")

PROJECT LOCATION:
c:\Users\palar\OneDrive\Desktop\InvoiceIQ

IMPORTANT FILE CHECK
README.md: EXISTS
notebooks/InvoiceIQ_Week3_Feature_Engineering.ipynb: EXISTS
reports/Week3_Feature_Engineering_Report.md: EXISTS
data/processed/X_train.npz: EXISTS
data/processed/X_validation.npz: EXISTS
data/processed/X_test.npz: EXISTS
data/processed/amount_bucket_config.joblib: EXISTS

PROCESSED FILES
amount_bucket_config.joblib: 0.00 MB
chronological_split_summary.csv: 0.00 MB
detailed_feature_catalogue.csv: 0.02 MB
feature_catalogue.csv: 0.01 MB
leakage_audit.csv: 0.00 MB
preprocessor.joblib: 0.01 MB
sales_rep_frequency_map.joblib: 0.01 MB
X_test.npz: 30.11 MB
X_train.npz: 56.01 MB
X_validation.npz: 29.35 MB
y_test.npy: 2.61 MB
y_train.npy: 5.19 MB
y_validation.npy: 2.57 MB

.GITIGNORE CONTENT
﻿.venv/
__pycache__/
.ipynb_checkpoints/

data/raw/
data/cleaned/*.csv


GIT STATUS
 M README.md
?? data/
?? notebooks/InvoiceIQ_Week3_Feature_Engineering.ipynb
?? reports/InvoiceIQ_Week2_EDA_Report.md
?? repo

In [94]:
from pathlib import Path
import subprocess

project_path = Path.cwd()

# 1. List data directory contents
print("DATA DIRECTORY CONTENTS\n")

for folder in ["data/raw", "data/cleaned", "data/processed"]:
    folder_path = project_path / folder

    print(f"\n{folder}/")

    if folder_path.exists():
        for path in sorted(folder_path.rglob("*")):
            if path.is_file():
                size_mb = path.stat().st_size / (1024 * 1024)
                print(f"  {path.relative_to(project_path)} ({size_mb:.2f} MB)")
    else:
        print("  Folder not found.")

# 2. Check important ignore rules
print("\nGIT IGNORE VERIFICATION\n")

paths_to_check = [
    ".venv/",
    "data/raw/train_invoices.csv",
    "data/raw/test_invoices.csv",
    "data/cleaned/InvoiceIQ_Cleaned_Data.csv",
    "data/processed/X_train.npz",
]

for path in paths_to_check:
    result = subprocess.run(
        ["git", "check-ignore", "-v", path],
        cwd=project_path,
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print(f"IGNORED: {path}")
        print("  Rule:", result.stdout.strip())
    else:
        print(f"NOT IGNORED: {path}")

# 3. Check ignored Git status
print("\nGIT IGNORED STATUS\n")

result = subprocess.run(
    ["git", "status", "--short", "--ignored"],
    cwd=project_path,
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else "No status entries.")

print("\nGIT SAFETY CHECK COMPLETED.")

DATA DIRECTORY CONTENTS


data/raw/
  data\raw\test_invoices.csv (573.34 MB)
  data\raw\train_invoices.csv (715.68 MB)

data/cleaned/
  data\cleaned\InvoiceIQ_Cleaned_Data.csv (624.88 MB)

data/processed/
  data\processed\amount_bucket_config.joblib (0.00 MB)
  data\processed\chronological_split_summary.csv (0.00 MB)
  data\processed\detailed_feature_catalogue.csv (0.02 MB)
  data\processed\feature_catalogue.csv (0.01 MB)
  data\processed\leakage_audit.csv (0.00 MB)
  data\processed\preprocessor.joblib (0.01 MB)
  data\processed\sales_rep_frequency_map.joblib (0.01 MB)
  data\processed\X_test.npz (30.11 MB)
  data\processed\X_train.npz (56.01 MB)
  data\processed\X_validation.npz (29.35 MB)
  data\processed\y_test.npy (2.61 MB)
  data\processed\y_train.npy (5.19 MB)
  data\processed\y_validation.npy (2.57 MB)

GIT IGNORE VERIFICATION

IGNORED: .venv/
  Rule: .gitignore:1:.venv/	.venv/
IGNORED: data/raw/train_invoices.csv
  Rule: .gitignore:5:data/raw/	data/raw/train_invoices.csv
IGNORE